# Notebook 07 — Report figures and numbers

Draws every thesis figure and headline number from the **saved outputs** of Notebooks 04–09b — nothing is
re-fitted here. Figures R1–R8 are the Results set; the supporting figures cover methodology and appendices;
the closing cells emit the write-up numbers and the monitored-station appendix table. The corridor-upgrade
policy scenario is its own notebook (Notebook 08).

**Inputs:** the frozen Stage-3 outputs (`stage3_final.json`, `od_final_t.csv`, `stage3_link_fit.csv`,
`A_low.npz`, `A_high.npz`, `A_neutral.npz`), `lowstress_route_stats.csv`, `pi_objective_curves.csv`, the
robustness CSVs, and the cached path stores. **Outputs:** the `fig_*` report figures, `results_numbers.txt`,
and the monitored-station appendix table/CSV/TeX.


In [2]:
import json, warnings
import numpy as np, pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patheffects import withStroke
import geopandas as gpd, osmnx as ox, networkx as nx
from scipy import sparse
import thesis_common as tc



tc.set_style()
HALO = [withStroke(linewidth=2.6, foreground="white")]
R1_INCLUDE_STAGE2 = True     # R1 as 4 panels (seed, Stage 1, Stage 2, Stage 3); False -> 3 panels
RUN_POLICY = True            # scenario figure (re-routes the low-stress class; slow)
POLICY_CORRIDOR = ["Oxford Street", "King Street"]
N_JOBS = 1                   # path stores are cached by NB05; only the policy scenario routes
NUM = []                     # lines for results_numbers.txt
def say(s=""):
    print(s); NUM.append(s)

## Section A — Load the saved outputs

Load the frozen model, the per-stage modelled flows, the monitored-link geometry and labels, and the
network/cordon, with assertions that the on-disk split and flows still match the frozen run. Defines the
shared map helpers (`basemap`, `cordon_line`, `frame`, `place_labels`) used by the figures.


In [3]:
final  = tc.read_final()
pi_star = final["pi_star"]
cal    = pd.read_csv(tc.DATA_PROCESSED / "od_final_t.csv").sort_values("pair_idx").reset_index(drop=True)
links  = pd.read_csv(tc.DATA_PROCESSED / "A_neutral_links.csv").sort_values("link_idx").reset_index(drop=True)
fit    = pd.read_csv(tc.DATA_PROCESSED / "stage3_link_fit.csv").sort_values("link_idx").reset_index(drop=True)
snap   = pd.read_csv(tc.DATA_PROCESSED / "count_link_snap.csv")
A_low  = sparse.load_npz(tc.DATA_PROCESSED / "A_low.npz").tocsr()
A_high = sparse.load_npz(tc.DATA_PROCESSED / "A_high.npz").tocsr()
A_neu  = sparse.load_npz(tc.DATA_PROCESSED / "A_neutral.npz").tocsr()
od_snapped = pd.read_csv(tc.DATA_PROCESSED / "jtw_2016_bike_od_snapped.csv")
L, P = len(links), len(cal)
v_obs  = links["v_obs"].to_numpy(float)
n_seed, t_cal, t_star = (cal[c].to_numpy(float) for c in ("n_seed", "t", "t_final"))
train  = tc.load_train_mask(L)
test   = fit["split"].eq("test").to_numpy()        # split recorded by NB05's single evaluation
assert (test == ~train).all(), "split in stage3_link_fit.csv disagrees with train_mask.npy"
assert abs(t_star.sum() - final["t_star_sum"]) < 1e-6, "od_final_t.csv does not match the frozen t*"
PI0 = tc.PI_LOW0

f = {  # modelled flow on monitored links, per stage
    "seed":   np.asarray(A_neu.dot(n_seed)).ravel(),
    "stage1": np.asarray(A_neu.dot(t_cal)).ravel(),
    "stage2": tc.flow(A_low, A_high, PI0, t_cal),
    "stage3": tc.flow(A_low, A_high, pi_star, t_star),
}
M = {k: tc.metrics(v[train], v_obs[train]) for k, v in f.items()}
M["stage3_test"] = tc.metrics(f["stage3"][test], v_obs[test])
assert np.allclose(f["stage3"], fit["f_final"]), "flows differ from NB07 - re-run NB07"

# geometry: one projected point per monitored link + names
names = tc.link_names(links, snap)
geo = snap.merge(links[["link_idx", "u", "v", "k"]], on=["u", "v", "k"]).groupby("link_idx").agg(lon=("lon", "mean"), lat=("lat", "mean"))
pts = gpd.GeoSeries(gpd.points_from_xy(geo.lon, geo.lat), crs=tc.LONLAT_CRS).to_crs(tc.PROJECTED_CRS)
sx = pd.Series(pts.x.values, index=geo.index).reindex(range(L)).to_numpy()
sy = pd.Series(pts.y.values, index=geo.index).reindex(range(L)).to_numpy()
label = np.array([names.get(i, (f"link {i}", None))[0] for i in range(L)], dtype=object)

Gr = tc.load_network(); tc.set_costs(Gr, lam=tc.STRESS_PENALTY)
edges = ox.graph_to_gdfs(Gr, nodes=False).reset_index()
cordon = gpd.read_file(tc.STUDY_AREA_CORDON).to_crs(tc.PROJECTED_CRS)
cx0, cy0, cx1, cy1 = cordon.total_bounds
print(f"pi* = {pi_star:.2f} | L={L} ({int(train.sum())} train / {int(test.sum())} held-out) | P={P:,}")

def place_labels(ax, xs, ys, texts, fs=8, color="black"):
    """Annotate points, choosing for each label the offset direction farthest from labels
    already placed (simple overlap avoidance; works in data units)."""
    x0, x1 = ax.get_xlim(); y0, y1 = ax.get_ylim(); dx, dy = 0.02 * (x1 - x0), 0.025 * (y1 - y0)
    opts = [(dx, dy, "left", "bottom"), (dx, -dy, "left", "top"), (-dx, dy, "right", "bottom"),
            (-dx, -dy, "right", "top"), (0, 2.2 * dy, "center", "bottom"), (0, -2.2 * dy, "center", "top")]
    placed = []
    for x, y, t in zip(xs, ys, texts):
        def score(o):
            px, py = x + o[0] * 3, y + o[1] * 1.5
            return min([np.hypot((px - a) / (x1 - x0), (py - b) / (y1 - y0)) for a, b in placed] + [9])
        o = max(opts, key=score)
        px, py = x + o[0], y + o[1]
        placed.append((x + o[0] * 3, y + o[1] * 1.5))
        ax.annotate(t, (x, y), xytext=(px, py), textcoords="data", ha=o[2], va=o[3], fontsize=fs, color=color,
                    path_effects=HALO, zorder=9, arrowprops=dict(arrowstyle="-", color="#666", lw=0.6))

def basemap(ax, elow=False, lw=0.25):
    edges.plot(ax=ax, color=tc.COL["grid"], lw=lw, zorder=0)
    if elow:
        edges[edges["lts"] <= tc.LOW_STRESS_MAX_LTS].plot(ax=ax, color=tc.COL["elow"], lw=0.7, zorder=1)
def cordon_line(ax):
    cordon.boundary.plot(ax=ax, color=tc.COL["cordon"], lw=0.9, ls="--", zorder=2)
def frame(ax, pad=400):
    ax.set_xlim(cx0 - pad, cx1 + pad); ax.set_ylim(cy0 - pad, cy1 + pad)
    ax.set_aspect("equal"); ax.set_axis_off()

pi* = 0.64 | L=77 (53 train / 24 held-out) | P=1,850


## R1 — Fit progression on the monitored stations

Observed vs modelled flow at each stage (seed → Stage 1 → Stage 2 → Stage 3), training points plus the
held-out diamonds at the final panel, each with its metric box.


In [4]:
panels = [("seed", "(a) Seed $n$, neutral AON"), ("stage1", "(b) Stage 1: GLS demand $t$")]
if R1_INCLUDE_STAGE2:
    panels.append(("stage2", f"(c) Stage 2: two-class at $\\pi^{{(0)}}$={PI0:.2f}"))
panels.append(("stage3", f"({'d' if R1_INCLUDE_STAGE2 else 'c'}) Stage 3: final $\\pi^*$={pi_star:.2f}"))
lim = 1.06 * max(v_obs.max(), *(f[k][train].max() for k, _ in panels), f["stage3"].max())
nr, nc = (2, 2) if len(panels) == 4 else (1, 3)
fig, axes = plt.subplots(nr, nc, figsize=(10, 10) if nr == 2 else (15, 5.2), sharex=True, sharey=True)
for ax, (k, ttl) in zip(np.ravel(axes), panels):
    ax.plot([0, lim], [0, lim], color=tc.COL["neutral"], ls="--", lw=0.9, zorder=1)
    ax.scatter(v_obs[train], f[k][train], s=30, c=tc.COL["train"], edgecolor="white", lw=0.4, zorder=3,
               label="training")
    if k == "stage3":
        ax.scatter(v_obs[test], f[k][test], s=42, c=tc.COL["test"], marker="D", edgecolor="black", lw=0.4,
                   zorder=4, label="held-out")
        tc.metric_box(ax, M["stage3"], title="training", color=tc.COL["train"])
        tc.metric_box(ax, M["stage3_test"], loc="lower right", title="held-out", color="#b35806")
        ax.legend(loc="center right", fontsize=8.5)
    else:
        tc.metric_box(ax, M[k], title="training")
    ax.set_title(ttl, fontsize=10.5); ax.set_xlim(0, lim); ax.set_ylim(0, lim)
for ax in np.ravel(axes)[-nc:]: ax.set_xlabel("observed flow (trips/h)")
for ax in np.ravel(axes)[::nc]: ax.set_ylabel("modelled flow (trips/h)")
fig.suptitle("Monitored-station fit from the seed through the three stages (dashed line: $y=x$)", y=0.995 if nr == 2 else 1.03)
fig.tight_layout()
tc.save_report(fig, "fig_R1_fit_progression")

  saved report figure  fig_R1_fit_progression.pdf/.png


## R2 — Two O–D pairs: stress-tolerant vs low-stress route

Picks two illustrative O–D pairs (recognisable corridor, meaningful separated-track length, decent demand)
and draws the stress-tolerant (fastest-time) route against the low-stress route, with the separated-track
segments highlighted.


In [5]:
rs = pd.read_csv(tc.DATA_PROCESSED / "lowstress_route_stats.csv")
exp_pairs, assembly, rep_idx = tc.expand_pairs(cal)
store_high = tc.route_store(tc.to_simple(Gr, "cost_high"), exp_pairs, tc.K_STORE, "cost_high", tc.store_tag("high"), n_jobs=N_JOBS)
store_low  = tc.route_store(tc.to_simple(Gr, "cost_low"),  exp_pairs, tc.K_STORE, "cost_low",  tc.store_tag("low"),  n_jobs=N_JOBS)
Gs_h, Gs_l = tc.to_simple(Gr, "cost_high"), tc.to_simple(Gr, "cost_low")
poly = cordon.union_all()
from shapely.geometry import LineString
def line_of(p): return LineString([(Gr.nodes[int(n)]["x"], Gr.nodes[int(n)]["y"]) for n in p])

EXCLUDE_STREETS = ["New South Head"]                         # network artefact
PREFER_STREETS  = ["pyrmont bridge", "king st", "kent st", "oxford st",      # recognisable, high-volume
                   "college st", "liverpool st", "union st", "harbour br"]
def _edge_d(Gs, a, b): return Gr.get_edge_data(int(a), int(b))[Gs[int(a)][int(b)]["key"]]
def _is_track(d):
    fac = str(d.get("facility_type", "")).lower()
    hwy = d.get("highway"); hwy = str(hwy[0] if isinstance(hwy, list) else hwy).lower()
    return ("track" in fac) or (hwy == "cycleway")
def route_track_m(Gs, p):
    return sum(Gs[int(a)][int(b)]["len_m"] for a, b in zip(p[:-1], p[1:]) if _is_track(_edge_d(Gs, a, b)))
def route_hits_street(Gs, p, needle):
    nl = needle.lower()
    return any((nm := tc._name(_edge_d(Gs, a, b))) and nl in nm.lower() for a, b in zip(p[:-1], p[1:]))
def route_named(Gs, p):                                     # {street name: length on route}
    d = {}
    for a, b in zip(p[:-1], p[1:]):
        nm = tc._name(_edge_d(Gs, a, b))
        if nm:
            nm = tc.abbreviate(nm); d[nm] = d.get(nm, 0) + Gs[int(a)][int(b)]["len_m"]
    return d
def dom_street(Gs, p):                                      # longest named street on the route
    d = route_named(Gs, p); return max(d, key=d.get) if d else None
def prefer_hit(Gs, pl, ph):                                 # which recognisable corridor the pair touches
    names = " ".join(list(route_named(Gs, pl)) + list(route_named(Gs, ph))).lower()
    return next((s for s in PREFER_STREETS if s in names), None)

def pick_pairs(det_lo, det_hi, min_sep, min_track_m=200, inside=0.8):
    c = rs[(~rs.same_route) & rs.detour_dist.between(det_lo, det_hi) & (rs.len_high_m > 1500)
           & rs.movement.isin(["inbound_internal", "inbound_external", "outbound"])].copy()
    cand = []
    for r in c.itertuples():
        ri = int(rep_idx[r.pair_idx])
        if ri < 0 or not store_low[ri] or not store_high[ri]:
            continue
        pl, ph = store_low[ri][0], store_high[ri][0]
        if any(route_hits_street(Gs_l, pl, s) or route_hits_street(Gs_h, ph, s) for s in EXCLUDE_STREETS):
            continue
        ll, lh = line_of(pl), line_of(ph)
        if ll.intersection(poly).length / ll.length < inside or lh.intersection(poly).length / lh.length < inside:
            continue
        cand.append(dict(r=r, pl=pl, ph=ph, mid=np.array(ll.interpolate(0.5, normalized=True).coords[0]),
                         track_m=route_track_m(Gs_l, pl), t_final=t_star[r.pair_idx],
                         dom=dom_street(Gs_l, pl), pref=prefer_hit(Gs_l, pl, ph)))
    # rank: touches a recognisable corridor first, then separated-track length, then demand
    cand.sort(key=lambda q: (q["pref"] is not None, q["track_m"] >= min_track_m, q["track_m"], q["t_final"]), reverse=True)
    out = []
    for q in cand:
        if any(o["dom"] == q["dom"] for o in out):                      # two different dominant streets
            continue
        if out and q["pref"] and q["pref"] == out[0]["pref"]:           # two different named corridors
            continue
        if all(np.hypot(*(q["mid"] - o["mid"])) > min_sep for o in out):
            out.append(q)
        if len(out) == 2:
            break
    return out

for args in [(5, 35, 1500, 300), (5, 45, 1200, 200), (3, 60, 800, 100), (1, 90, 300, 0)]:
    picks = pick_pairs(*args)
    if len(picks) == 2 and picks[0]["track_m"] > 0:
        break
print(f"  R2 pairs chosen {args}: pair_idx {[p_['r'].pair_idx for p_ in picks]}, "
      f"track_m {[round(p_['track_m']) for p_ in picks]}, "
      f"dominant {[p_['dom'] for p_ in picks]}, corridor {[p_['pref'] for p_ in picks]}")

# --- street labels for each route ---
def street_runs(G, p, min_frac=0.10, max_n=4):
    runs, cur = [], None
    for a, b in zip(p[:-1], p[1:]):
        d = Gr.get_edge_data(int(a), int(b)); d = d[G[int(a)][int(b)]["key"]]
        nm = tc._name(d); L_ = d["len_m"]
        nm = tc.abbreviate(nm) if nm else None
        xm = (Gr.nodes[int(a)]["x"] + Gr.nodes[int(b)]["x"]) / 2; ym = (Gr.nodes[int(a)]["y"] + Gr.nodes[int(b)]["y"]) / 2
        if cur and cur["name"] == nm:
            cur["len"] += L_; cur["pts"].append((xm, ym, L_))
        else:
            if cur: runs.append(cur)
            cur = dict(name=nm, len=L_, pts=[(xm, ym, L_)])
    if cur: runs.append(cur)
    tot = sum(r["len"] for r in runs) or 1
    runs = [r for r in runs if r["name"] and r["len"] >= min_frac * tot]
    merged = {}
    for r in runs:                                  # one label per street name
        if r["name"] not in merged or r["len"] > merged[r["name"]]["len"]: merged[r["name"]] = r
    out = sorted(merged.values(), key=lambda r: -r["len"])[:max_n]
    for r in out:
        w = np.array([q[2] for q in r["pts"]]); xy = np.array([q[:2] for q in r["pts"]])
        r["xy"] = xy[np.argmin(np.abs(np.cumsum(w) - w.sum() / 2))]
    return out

if len(picks) < 2:
    print(f"  only {len(picks)} pair(s) met the R2 criteria - relax the filters above")
TRACK_COL = "#e6a500"                               # amber: separated-track segments
fig, axes = plt.subplots(1, max(len(picks), 1), figsize=(14, 7.2))
axes = np.atleast_1d(axes)
for ax, pk in zip(axes, picks):
    r, pl, ph = pk["r"], pk["pl"], pk["ph"]
    xs = [Gr.nodes[int(n)]["x"] for n in np.concatenate([pl, ph])]; ys = [Gr.nodes[int(n)]["y"] for n in np.concatenate([pl, ph])]
    pad = 0.15 * max(np.ptp(xs), np.ptp(ys)) + 150
    ax.set_xlim(min(xs) - pad, max(xs) + pad); ax.set_ylim(min(ys) - pad, max(ys) + pad)
    basemap(ax, elow=True, lw=0.4)
    for p, c, lw_, z in [(ph, tc.COL["high"], 5.0, 4), (pl, tc.COL["low"], 2.8, 5)]:
        ax.plot(*line_of(p).xy, color=c, lw=lw_, alpha=0.9, solid_capstyle="round", zorder=z)
    for a, b in zip(pl[:-1], pl[1:]):               # amber halo on the separated-track segments
        if _is_track(_edge_d(Gs_l, a, b)):
            ax.plot([Gr.nodes[int(a)]["x"], Gr.nodes[int(b)]["x"]], [Gr.nodes[int(a)]["y"], Gr.nodes[int(b)]["y"]],
                    color=TRACK_COL, lw=6.5, alpha=0.9, solid_capstyle="round", zorder=4.5)
    for nlab, n in (("O", pl[0]), ("D", pl[-1])):
        ax.scatter(Gr.nodes[int(n)]["x"], Gr.nodes[int(n)]["y"], s=70, c="black", zorder=7)
        ax.annotate(nlab, (Gr.nodes[int(n)]["x"], Gr.nodes[int(n)]["y"]), xytext=(6, 6),
                    textcoords="offset points", fontsize=11, fontweight="bold", path_effects=HALO, zorder=8)
    placed, named = [], set()
    for p, G, c in [(pl, Gs_l, "#0b5a2a"), (ph, Gs_h, "#7f1d12")]:
        for run in street_runs(G, p):
            if run["name"] in named: continue          # shared streets labelled once
            if any(np.hypot(*(run["xy"] - q)) < 0.06 * max(np.ptp(xs), np.ptp(ys)) for q in placed): continue
            placed.append(run["xy"]); named.add(run["name"])
            ax.annotate(run["name"], run["xy"], fontsize=8.5, color=c, ha="center", va="center",
                        path_effects=HALO, zorder=9)
    ax.set_title(f"{r.len_high_m/1000:.1f} km pair: low-stress route +{r.detour_dist:.0f}% distance "
                 f"(+{r.detour_time:.0f}% time)\nLTS 1 share {r.share_lts1_low:.0f}% vs {r.share_lts1_high:.0f}% "
                 f"on the fastest route", fontsize=10)
    ax.set_aspect("equal"); ax.set_axis_off()
fig.legend(handles=[Line2D([], [], color=tc.COL["high"], lw=4, label="stress-tolerant route (fastest time)"),
                    Line2D([], [], color=tc.COL["low"], lw=3, label="low-stress route"),
                    Line2D([], [], color=TRACK_COL, lw=4, label="separated cycle track on low-stress route"),
                    Line2D([], [], color=tc.COL["elow"], lw=2, label="LTS 1 network")],
           loc="lower center", ncol=4, bbox_to_anchor=(0.5, -0.02))
x0, x1 = ax.get_xlim(); y0, y1 = ax.get_ylim()
ax.annotate("N", xy=(x0 + (x1 - x0) * 0.95, y0 + (y1 - y0) * 0.90),
            xytext=(x0 + (x1 - x0) * 0.95, y0 + (y1 - y0) * 0.84),
            ha="center", va="center", fontsize=12, fontweight="bold",
            arrowprops=dict(arrowstyle="-|>", color="black", lw=1.6), zorder=10)
fig.tight_layout(rect=(0, 0.04, 1, 1))
tc.save_report(fig, "fig_R2_route_pairs")

  [high_beta0_grade_k5_sprd5] loaded cached path store (8,721 pairs, k=5)
  [low_lam2.4_beta0_grade_k5_sprd5] loaded cached path store (8,721 pairs, k=5)
  R2 pairs chosen (5, 35, 1500, 300): pair_idx [1296, 780], track_m [10331, 5626], dominant ['Bourke St', 'Anzac Pde'], corridor ['king st', 'kent st']
  saved report figure  fig_R2_route_pairs.pdf/.png


## R3 — Low-stress routing behaviour

Distributions across all pairs: (a) the share of each route's length on LTS 1 (low-stress vs fastest), and
(b) the distance detour of the low-stress route relative to the shortest path, against the Broach et al.
(2012) +11% benchmark.


In [6]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# (a) share of each route on LTS 1: low-stress vs fastest
ax = axes[0]
bins = np.arange(0, 101, 5)
ax.hist(rs.share_lts1_low,  bins=bins, color=tc.COL["low"], alpha=0.9, label="low-stress route")
ax.hist(rs.share_lts1_high, bins=bins, histtype="step", lw=1.6, color=tc.COL["neutral"], label="fastest route")
m_low = np.nanmedian(rs.share_lts1_low)
ax.axvline(m_low, color="black", lw=1.4)
ax.text(m_low - 2, ax.get_ylim()[1] * 0.80, f"low-stress\nmedian {m_low:.0f}%", ha="right", va="top",
        fontsize=9, path_effects=HALO)
ax.set_xlabel("share of route length on LTS 1 (%)"); ax.set_ylabel("O-D pairs")
ax.set_title("(a) Share of route length on LTS 1")
ax.legend(loc="upper left", fontsize=8)

# (b) distance detour relative to the shortest path
ax = axes[1]
ax.hist(np.clip(rs.detour_dist, 0, 40), bins=np.arange(0, 41, 2), color=tc.COL["low"], alpha=0.9)
m_det = np.nanmedian(rs.detour_dist); top = ax.get_ylim()[1]
ax.axvline(m_det, color="black", lw=1.6)
ax.text(m_det + 1, top * 0.96, f"median +{m_det:.0f}%", va="top", fontsize=9, path_effects=HALO)
ax.axvline(11, color=tc.COL["obs"], ls="--", lw=1.4)
ax.text(11 + 1, top * 0.66, "Broach et al.\n(2012): +11%", color=tc.COL["obs"], fontsize=9, path_effects=HALO)
ax.set_xlabel("extra distance vs shortest path (%)"); ax.set_ylabel("O-D pairs")
ax.set_title("(b) Distance detour relative to the shortest path")

fig.suptitle("Low-stress class routing behaviour", y=1.02)
fig.tight_layout()
tc.save_report(fig, "fig_R3_lowstress_behaviour")

  saved report figure  fig_R3_lowstress_behaviour.pdf/.png


## R4 — Share objective (training stations)

The mean-GEH objective as a continuous function of $\pi_{low}$, with the rescaled RMSE and GEH-pass curves,
marking the Dill $\pi^{(0)}$ and the selected $\pi^*$.


In [7]:
oc = pd.read_csv(tc.DATA_PROCESSED / "pi_objective_curves.csv")
fig, ax = plt.subplots(figsize=(8, 4.6))
ax2 = ax.twinx(); ax2.spines["right"].set_visible(True)
nrm = lambda a: (a - a.min()) / (np.ptp(a) or 1)
ax2.plot(oc.pi, nrm(oc.rmse_final), color="#6a8caf", lw=1.3, alpha=0.55, label="RMSE (rescaled)")
ax2.plot(oc.pi, nrm(100 - oc.geh_pass_final), color=tc.COL["test"], lw=1.3, alpha=0.55, drawstyle="steps-mid",
         label="% failing GEH$<$5 (rescaled)")
ax2.set_ylabel("faint lines: rescaled, 0 = best, 1 = worst", color="#777"); ax2.set_ylim(-0.05, 1.05)
ax.plot(oc.pi, oc.mean_geh_final, color="black", lw=2.3, zorder=5, label="mean GEH (selection objective)")
for pv, lab, mk in [(PI0, f"$\\pi^{{(0)}}$={PI0:.2f} (Dill)", "o"), (pi_star, f"$\\pi^*$={pi_star:.2f}", "o")]:
    yv = float(np.interp(pv, oc.pi, oc.mean_geh_final))
    ax.scatter([pv], [yv], s=70, zorder=6, facecolor="white" if pv == PI0 else tc.COL["low"], edgecolor="black")
    ax.annotate(lab, (pv, yv), xytext=(0, 14), textcoords="offset points", ha="center", fontsize=9.5,
                bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.9))
ax.set_zorder(ax2.get_zorder() + 1); ax.patch.set_visible(False)
ax.set_xlabel(r"low-stress share $\pi_{low}$"); ax.set_ylabel("mean GEH, training stations")
h1, l1 = ax.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
ax.legend(h1 + h2, l1 + l2, loc="upper center", bbox_to_anchor=(0.5, -0.16), ncol=3, fontsize=8.5)
ax.set_title("Mean GEH as a continuous function, GEH-pass count as a discrete step function")
tc.save_report(fig, "fig_R4_share_objective")

  saved report figure  fig_R4_share_objective.pdf/.png


## R5 — Network flow maps (one per stage)

Assigned peak-hour flow across the network at each stage on identical colour and width scales, with the
busiest named corridors numbered.


In [8]:
def link_flow_table(fl_dir):
    """directed {(u,v): flow} -> one row per edge geometry with two-way flow."""
    two = {}
    for (a, b), x in fl_dir.items():
        key = (min(a, b), max(a, b)); two[key] = two.get(key, 0.0) + x
    e = edges.copy()
    e["pair"] = [(min(a, b), max(a, b)) for a, b in zip(e.u, e.v)]
    e = e.drop_duplicates("pair")
    e["flow"] = e["pair"].map(two).fillna(0.0)
    return e

fl3 = tc.edge_flows_spread(store_low, tc.K_PATHS, pi_star * t_star, assembly)
for kk, x in tc.edge_flows_spread(store_high, tc.K_PATHS, (1 - pi_star) * t_star, assembly).items(): fl3[kk] += x
fl2 = tc.edge_flows_spread(store_low, tc.K_PATHS, PI0 * t_cal, assembly)
for kk, x in tc.edge_flows_spread(store_high, tc.K_PATHS, (1 - PI0) * t_cal, assembly).items(): fl2[kk] += x
s1p = tc.DATA_PROCESSED / "stage1_edge_flows.csv"
fl1 = None
if s1p.exists():
    s1 = pd.read_csv(s1p); fl1 = {(int(a), int(b)): x for a, b, x in zip(s1.u, s1.v, s1.flow_t)}
else:
    print("  stage1_edge_flows.csv missing (NB04 Section G2) - Stage-1 flow map skipped")
E = {"stage3": link_flow_table(fl3), "stage2": link_flow_table(fl2)}
if fl1 is not None: E["stage1"] = link_flow_table(fl1)
VMAX = float(np.percentile(E["stage3"].loc[E["stage3"].flow > 0, "flow"], 99))
norm = mpl.colors.Normalize(0, VMAX); cmap = mpl.colormaps[tc.FLOW_CMAP]

def numbered_corridors(ax, e, n_lab=10, fontsize=9):
    """Place circled numbers on the highest-flow named corridors and return the
    [(number, name)] list for the legend. Numbering is by total flow x length."""
    e = e[e.flow > 0].copy()
    e["nm"] = e["name"].map(lambda x: tc._name({"name": x}))
    e = e.dropna(subset=["nm"])
    e["fl_len"] = e.flow * e.len_m
    order = e.groupby("nm").fl_len.sum().sort_values(ascending=False).index[:n_lab]

    (x0, x1), (y0, y1) = ax.get_xlim(), ax.get_ylim()
    clr = 0.035 * (x1 - x0)                      # keep markers at least this far apart
    placed, legend, i = [], [], 0
    for nm in order:
        rows = e[e.nm == nm].sort_values("flow", ascending=False)
        pt = None
        for _, row in rows.head(6).iterrows():    # try points along the busiest segments
            for f in (0.5, 0.4, 0.6, 0.3, 0.7):
                c = row.geometry.interpolate(f, normalized=True)
                if all(np.hypot(c.x - px, c.y - py) > clr for px, py in placed):
                    pt = c; break
            if pt is not None:
                break
        if pt is None:
            pt = rows.iloc[0].geometry.interpolate(0.5, normalized=True)
        placed.append((pt.x, pt.y)); i += 1
        ax.annotate(str(i), (pt.x, pt.y), fontsize=fontsize, fontweight="bold",
                    ha="center", va="center", zorder=11, color="black",
                    bbox=dict(boxstyle="circle,pad=0.22", fc="white", ec="black", lw=1.1, alpha=0.95))
        legend.append((i, tc.abbreviate(nm)))
    return legend

STAGE_TITLE = {"stage1": "Stage 1: neutral AON of GLS demand $t$",
               "stage2": f"Stage 2: two classes at $\\pi^{{(0)}}$={PI0:.2f}",
               "stage3": f"Stage 3: calibrated flows at $\\pi^*$={pi_star:.2f}"}
for st, e in E.items():
    fig, ax = plt.subplots(figsize=(9, 9.6))
    basemap(ax)
    ef = e[e.flow > 0.5].sort_values("flow")
    ef.plot(ax=ax, column="flow", cmap=cmap, norm=norm, zorder=3,
            linewidth=0.4 + 3.6 * np.sqrt(np.clip(ef.flow, 0, VMAX) / VMAX))
    cordon_line(ax); frame(ax)                   # frame() first so marker spacing uses final limits
    legend = numbered_corridors(ax, e)

    # numbered legend (corridor names) + cordon boundary, placed below the map so nothing is hidden
    handles = [Line2D([], [], ls="none", marker="o", color="white", markersize=12,
                      markeredgecolor="black", markeredgewidth=1.1,
                      label=f"{n}.  {nm}") for n, nm in legend]
    handles.append(Line2D([], [], color=tc.COL["cordon"], ls="--", lw=1.1, label="Study-area cordon"))
    ax.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, -0.02),
              ncol=3, fontsize=8.5, title="Highest-flow corridors", title_fontsize=9,
              frameon=True, handletextpad=0.4, columnspacing=1.2)

    sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
    cb = fig.colorbar(sm, ax=ax, shrink=0.5, pad=0.01); cb.set_label("two-way peak-hour flow (trips/h)")
    x0, x1 = ax.get_xlim(); y0, y1 = ax.get_ylim()
    ax.annotate("N", xy=(x0 + (x1 - x0) * 0.95, y0 + (y1 - y0) * 0.90),
            xytext=(x0 + (x1 - x0) * 0.95, y0 + (y1 - y0) * 0.84),
            ha="center", va="center", fontsize=12, fontweight="bold",
            arrowprops=dict(arrowstyle="-|>", color="black", lw=1.6), zorder=10)
    ax.set_title(STAGE_TITLE[st] + f"\n(width and colour $\\propto$ flow; colour capped at {VMAX:.0f})", fontsize=11)
    tc.save_report(fig, "fig_R5_flow_map" + ("" if st == "stage3" else f"_{st}"))
top_corr = E["stage3"].assign(nm=E["stage3"]["name"].map(lambda x: tc._name({"name": x}))).dropna(subset=["nm"])
top_corr = (top_corr.assign(vkm=top_corr.flow * top_corr.len_m / 1000).groupby("nm").agg(max_flow=("flow", "max"), veh_km=("vkm", "sum"))
            .sort_values("veh_km", ascending=False).head(10))
print(top_corr.round(0))

  saved report figure  fig_R5_flow_map.pdf/.png
  saved report figure  fig_R5_flow_map_stage2.pdf/.png
  saved report figure  fig_R5_flow_map_stage1.pdf/.png
                                                 max_flow  veh_km
nm                                                               
Bourke Street                                       394.0   590.0
Anzac Parade                                        346.0   440.0
Oxford Street                                       480.0   409.0
ANZAC Bridge Shared Pedestrian and Bicycle Path     435.0   401.0
Kent Street                                         409.0   392.0
Lilyfield Road                                      292.0   370.0
Moore Park Road                                     394.0   351.0
Bourke Road                                         149.0   292.0
King Street                                         236.0   266.0
South Dowling Street                                195.0   261.0


## R6 — Station dumbbell: the 20 busiest stations through the stages

Dumbbell plot of observed vs modelled flow at the 20 busiest stations, showing how the estimate moves
across the stages.


In [9]:
N_TOP = 20
o = np.argsort(v_obs)[::-1][:N_TOP]
y = np.arange(N_TOP)[::-1]
fig, ax = plt.subplots(figsize=(9, 8.2))
for yi, i in zip(y, o):
    ax.plot([v_obs[i], f["stage3"][i]], [yi, yi], color="#c8c8c8", lw=2.2, zorder=1)
tr = train[o]
ax.scatter(f["stage1"][o][tr], y[tr], marker="x", s=30, color=tc.STAGE_COL["stage1"], zorder=2, label="Stage 1 (training only)")
ax.scatter(f["stage2"][o][tr], y[tr], marker="o", s=34, facecolor="white", edgecolor=tc.STAGE_COL["stage2"], zorder=2,
           label=f"Stage 2 at $\\pi^{{(0)}}$ (training only)")
ax.scatter(f["stage3"][o], y, s=52, c=np.where(tr, tc.COL["train"], tc.COL["test"]), edgecolor="black", lw=0.4, zorder=3)
ax.scatter(v_obs[o], y, marker="D", s=44, color=tc.COL["obs"], zorder=4, label="observed")
ax.set_yticks(y)
ax.set_yticklabels([label[i] for i in o], fontsize=8.5)
for tl, i in zip(ax.get_yticklabels(), o):
    tl.set_color(tc.COL["train"] if train[i] else "#b35806")
ax.set_xlabel("peak-hour flow (trips/h)")
hand = ax.get_legend_handles_labels()[0] + [
    Line2D([], [], marker="o", ls="", markerfacecolor=tc.COL["train"], markeredgecolor="black", label=f"Stage 3 at $\\pi^*$ (training)"),
    Line2D([], [], marker="o", ls="", markerfacecolor=tc.COL["test"], markeredgecolor="black", label=f"Stage 3 at $\\pi^*$ (held-out)")]
ax.legend(handles=hand, loc="lower right", fontsize=8.5, frameon=True)
ax.set_title(f"Observed vs modelled flow at the {N_TOP} busiest stations\n(label colour: blue = training, orange = held-out)")
ax.grid(axis="x", alpha=0.25)
tc.save_report(fig, "fig_R6_station_dumbbell")

  saved report figure  fig_R6_station_dumbbell.pdf/.png


## R7 — Signed residual maps (one per stage)

Modelled − observed residual at each stage on identical diverging scales, so over- and under-prediction
are comparable across stages.


In [10]:
from matplotlib.patheffects import withStroke

res = {k: f[k] - v_obs for k in ("stage1", "stage2", "stage3")}
RMAX = float(np.nanpercentile(np.abs(np.concatenate([res["stage1"][train], res["stage2"][train], res["stage3"]])), 98))
rnorm = mpl.colors.TwoSlopeNorm(vmin=-RMAX, vcenter=0, vmax=RMAX)
size = lambda v: 25 + 330 * v / v_obs.max()
STAGE_R7 = {"stage1": "Stage 1: neutral AON of $t$ - training stations",
            "stage2": f"Stage 2: two classes at $\\pi^{{(0)}}$={PI0:.2f} - training stations",
            "stage3": f"Stage 3: $\\pi^*$={pi_star:.2f} - training and held-out stations"}

# the 10 worst-GEH stations at stage 3 == the top-error links; label only these
_den = f["stage3"] + v_obs
_geh3 = np.sqrt(np.divide(2 * (f["stage3"] - v_obs) ** 2, _den, out=np.zeros_like(_den), where=_den > 0))
TOP_ERR = set(np.argsort(-_geh3)[:10].tolist())

def place_labels_clear(ax, xs, ys, texts, fontsize=8.5, base_pts=12):
    """Offset each label off its marker, avoid markers and other labels, draw a leader."""
    ax.figure.canvas.draw()
    bb = ax.get_window_extent()
    (X0, X1), (Y0, Y1) = ax.get_xlim(), ax.get_ylim()
    ppt = ax.figure.dpi / 72.0
    dux = (X1 - X0) / bb.width * ppt
    duy = (Y1 - Y0) / bb.height * ppt
    boxes = [(x - 4*dux, x + 4*dux, y - 4*duy, y + 4*duy) for x, y in zip(xs, ys)]
    clash = lambda b: any(b[0] < p[1] and p[0] < b[1] and b[2] < p[3] and p[2] < b[3] for p in boxes)
    offs = [(1, 0), (1, 1), (0, 1), (-1, 1), (-1, 0), (-1, -1), (0, -1), (1, -1)]
    for x, y, t in zip(xs, ys, texts):
        w, h = 0.58 * fontsize * len(t) * dux, 1.3 * fontsize * duy
        anchor = None
        for mult in (1.0, 1.8, 2.8, 4.0):
            for ox_, oy_ in offs:
                dx, dy = ox_ * base_pts * mult * dux, oy_ * base_pts * mult * duy
                ha = "left" if ox_ > 0 else ("right" if ox_ < 0 else "center")
                cx = x + dx + (w/2 if ox_ > 0 else (-w/2 if ox_ < 0 else 0))
                box = (cx - w/2, cx + w/2, y + dy - h/2, y + dy + h/2)
                if not clash(box):
                    anchor = (x + dx, y + dy, ha); boxes.append(box); break
            if anchor:
                break
        if anchor is None:
            anchor = (x + base_pts * 4 * dux, y, "left")
        ax.annotate(t, (x, y), xytext=(anchor[0], anchor[1]), ha=anchor[2], va="center",
                    fontsize=fontsize, zorder=21,
                    path_effects=[withStroke(linewidth=2.5, foreground="white")],
                    arrowprops=dict(arrowstyle="-", color="0.4", lw=0.6, shrinkA=0, shrinkB=2))

for st in ("stage1", "stage2", "stage3"):
    show = np.ones(L, bool) if st == "stage3" else train.copy()
    fig, ax = plt.subplots(figsize=(9, 9.6))
    basemap(ax); cordon_line(ax)
    ax.scatter(sx[show & train], sy[show & train], s=size(v_obs[show & train]), c=res[st][show & train],
               cmap=tc.RESID_CMAP, norm=rnorm, edgecolor="black", lw=0.4, zorder=5)
    if st == "stage3":
        ax.scatter(sx[test], sy[test], s=size(v_obs[test]), c=res[st][test], cmap=tc.RESID_CMAP, norm=rnorm,
                   edgecolor=tc.COL["test"], lw=1.8, zorder=5)
    frame(ax)

    # north arrow (top right)
    x0, x1 = ax.get_xlim(); y0, y1 = ax.get_ylim()
    ax.annotate("N", xy=(x0 + (x1 - x0) * 0.95, y0 + (y1 - y0) * 0.90),
        xytext=(x0 + (x1 - x0) * 0.95, y0 + (y1 - y0) * 0.84),
        ha="center", va="center", fontsize=12, fontweight="bold",
        arrowprops=dict(arrowstyle="-|>", color="black", lw=1.6), zorder=10)


    # label only the top-10 error stations that are visible in this panel
    lab = [i for i in TOP_ERR if show[i]]
    place_labels_clear(ax, sx[lab], sy[lab], [f"{label[i]} ({res[st][i]:+.0f})" for i in lab])

    # colour bar along the bottom, clear of the labels and the legend
    sm = mpl.cm.ScalarMappable(norm=rnorm, cmap=tc.RESID_CMAP)
    cb = fig.colorbar(sm, ax=ax, orientation="horizontal", fraction=0.046, pad=0.035, extend="both")
    cb.set_label("modelled $-$ observed (trips/h);  blue = under-predicted")

    m = M[st]; extra = f" | held-out mean GEH {M['stage3_test']['mean_geh']:.2f}" if st == "stage3" else ""
    ax.set_title(STAGE_R7[st] + f"\ntraining mean GEH {m['mean_geh']:.2f}{extra}; marker size $\\propto$ observed flow; labels = top-10 error links", fontsize=10.5)
    if st == "stage3":
        ax.legend(handles=[Line2D([], [], marker="o", ls="", markerfacecolor="white", markeredgecolor="black", label="training"),
                           Line2D([], [], marker="o", ls="", markerfacecolor="white", markeredgecolor=tc.COL["test"], markeredgewidth=2, label="held-out")],
                  loc="lower left", frameon=True)
    tc.save_report(fig, "fig_R7_residual_map" + ("" if st == "stage3" else f"_{st}"))

  saved report figure  fig_R7_residual_map_stage1.pdf/.png
  saved report figure  fig_R7_residual_map_stage2.pdf/.png
  saved report figure  fig_R7_residual_map.pdf/.png


## R8 — Robustness: range of $\pi^*$ per parameter

The span of $\pi^*$ across every robustness sweep (Notebooks 09a/09b), summarising how stable the share is.


In [11]:
# ── Robustness figures ────────────────────────────────────────────────────────
# Two separate figures:
#   fig_R8a_sweep_ranges  — how far pi* moves within each parameter sweep (dumbbell)
#   fig_R8b_stress_sweep  — the lambda_stress sweep in full (pi* and held-out R2 per value)

def load_robustness(fn):
    """Return the robustness CSV if it exists and has rows, else None."""
    path = tc.DATA_PROCESSED / fn
    if not path.exists():
        return None
    df = pd.read_csv(path)
    return df if len(df) else None

# Sweeps shown as ranges in the dumbbell figure, in display order (top to bottom).
SWEEPS = [
    (r"$\sigma_0^2$ (prior dispersion)",   "robustness_sigma2.csv"),
    (r"$W$ (count-variance window)",       "robustness_wwindow.csv"),
    (r"$k$ (paths per pair)",              "robustness_k.csv"),
    (r"$\beta_{grade}$ (climb penalty)",   "robustness_beta_grade.csv"),
    (r"grade ablation (flat speed)",       "robustness_flat.csv"),
]

sweeps = [(label, df) for label, fn in SWEEPS
          if (df := load_robustness(fn)) is not None]
stress = load_robustness("robustness_stress_penalty.csv")

if not sweeps and stress is None:
    print("  no robustness CSVs yet - run NB09a/NB09b, then re-run this cell")

# ── Figure A: range of pi* within each sweep ──────────────────────────────────
# One row per parameter. Grey bar = min..max of pi* over that sweep's settings;
# small dots = individual settings; large black dot = the baseline setting.
# The headline pi* is drawn as a reference line.
if sweeps:
    labels = [label for label, _ in sweeps]
    fig, ax = plt.subplots(figsize=(9, 0.9 * len(sweeps) + 1.8))

    for row, (label, df) in enumerate(sweeps):
        lo, hi = df.pi_star.min(), df.pi_star.max()
        ax.plot([lo, hi], [row, row], color="#9e9e9e", lw=9,
                solid_capstyle="round", alpha=0.55, zorder=1)
        ax.scatter(df.pi_star, [row] * len(df), s=18, color="#555555", zorder=2)

        baseline = df[df.is_baseline.astype(bool)]
        if len(baseline):
            ax.scatter(baseline.pi_star, [row] * len(baseline),
                       s=70, color="black", zorder=3)

        # Range + held-out R2, printed just under each bar so nothing hangs off the axis.
        ax.text(hi + 0.01, row,
                f"{lo:.2f}-{hi:.2f}  (R$^2$ {df.test_r2.min():.2f}-{df.test_r2.max():.2f})",
                va="center", fontsize=8, color="#666")

    ax.axvline(pi_star, color=tc.COL["low"], ls="--", lw=1.4, zorder=0)
    ax.text(pi_star, len(sweeps) - 0.4, f" $\\pi^*$={pi_star:.2f}",
            color=tc.COL["low"], fontsize=9, va="bottom")

    ax.set_yticks(range(len(sweeps)))
    ax.set_yticklabels(labels)
    ax.set_ylim(-0.7, len(sweeps) - 0.3)
    ax.set_xlim(0, 1)
    ax.set_xlabel(r"converged $\pi^*_{low}$")
    ax.set_title("Range of $\\pi^*$ across each sweep (black dot = baseline)", loc="left")
    tc.save_report(fig, "fig_R8a_sweep_ranges")

# ── Figure B: the lambda_stress sweep ─────────────────────────────────────────
# pi* at each lambda value (baseline highlighted), with held-out R2 annotated below.
if stress is not None:
    x = np.arange(len(stress))
    fig, ax = plt.subplots(figsize=(6, 4))

    ax.plot(x, stress.pi_star, "o-", color="black", zorder=2)
    is_base = stress.is_baseline.astype(bool).to_numpy()
    ax.scatter(x[is_base], stress.pi_star[is_base],
               s=80, color=tc.COL["low"], zorder=3, label="baseline")
    ax.axhline(pi_star, color=tc.COL["low"], ls="--", lw=1, zorder=1)

    for xi, row in zip(x, stress.itertuples()):
        ax.annotate(f"R$^2$ {row.test_r2:.2f}", (xi, row.pi_star),
                    xytext=(0, -14), textcoords="offset points",
                    ha="center", fontsize=7.5, color="#666")

    ax.set_xticks(x)
    ax.set_xticklabels([f"{v:g}" for v in stress.value])
    ax.set_ylim(-0.05, 1.05)
    ax.set_xlabel(r"$\lambda_{stress}$")
    ax.set_ylabel(r"$\pi^*_{low}$")
    ax.set_title("$\\lambda_{stress}$ sweep (held-out $R^2$ shown below each point)", loc="left")
    ax.legend(loc="lower right", fontsize=8, frameon=False)
    tc.save_report(fig, "fig_R8b_stress_sweep")

  saved report figure  fig_R8a_sweep_ranges.pdf/.png
  saved report figure  fig_R8b_stress_sweep.pdf/.png


## Supporting figures (methodology + appendices)

The remaining methodology and appendix figures assembled from the saved outputs (network/LTS context,
count-station map, demand provenance, and the split carriageway check).


In [12]:
# S1 — component ladder (training stations for every stage, held-out only at Stage 3)
rows = [("Seed $n$, neutral AON", M["seed"], "train"), ("Stage 1: GLS $t$", M["stage1"], "train"),
        ("Stage 2: two-class at $\\pi^{(0)}$", M["stage2"], "train"), ("Stage 3: $\\pi^*$", M["stage3"], "train"),
        ("Stage 3: $\\pi^*$ (held-out)", M["stage3_test"], "test")]
lad = pd.DataFrame([dict(configuration=r[0], sample=r[2], **{k: r[1][k] for k in ("n", "mean_geh", "geh_pass", "rmse", "r2", "r2_cod", "bias_pct")}) for r in rows])
lad.to_csv(tc.FIG_REPORT / "results_component_ladder.csv", index=False)
fig, ax = plt.subplots(figsize=(8.5, 3.8))
yy = np.arange(len(lad))[::-1]
ax.barh(yy, lad.mean_geh, color=[tc.COL["train"] if s == "train" else tc.COL["test"] for s in lad["sample"]])
for yi, r in zip(yy, lad.itertuples()):
    ax.text(r.mean_geh + 0.05, yi, f"{r.mean_geh:.2f}   (GEH$<$5 {r.geh_pass:.0f}%, $R^2$ {r.r2:.2f})", va="center", fontsize=8.5)
ax.set_yticks(yy); ax.set_yticklabels(lad.configuration); ax.set_xlim(0, lad.mean_geh.max() * 1.6)
ax.set_xlabel("mean GEH (lower is better)"); ax.set_title("Component ladder (blue: training stations; orange: held-out)")
tc.save_report(fig, "fig_S1_component_ladder")

# S2 — demand by movement through the stages
mv = ["inbound_internal", "inbound_external", "outbound", "through"]
comp = cal.groupby("movement")[["n_seed", "t", "t_final"]].sum().reindex(mv).fillna(0)
fig, ax = plt.subplots(figsize=(9, 4.4)); x = np.arange(len(mv)); w = 0.27
for j, (c, lab, col) in enumerate([("n_seed", "seed $n$", tc.STAGE_COL["seed"]), ("t", "Stage 1 $t$", tc.STAGE_COL["stage1"]),
                                   ("t_final", "final $t^*$", "black")]):
    b = ax.bar(x + (j - 1) * w, comp[c], w, color=col, label=lab)
    for bb, v in zip(b, comp[c]): ax.text(bb.get_x() + w / 2, v, f"{v:,.0f}", ha="center", va="bottom", fontsize=7.5)
ax.set_xticks(x); ax.set_xticklabels(["inbound\n(internal)", "inbound\n(external)", "outbound", "through"])
ax.set_ylabel("peak-hour trips"); ax.legend(); ax.set_title("Demand by movement class through the stages")
tc.save_report(fig, "fig_S2_demand_by_movement")

# S3 — adjustment factors through the stages
pos = n_seed > 0
fig, ax = plt.subplots(figsize=(7.5, 4))
bins = np.linspace(0, 3, 41)
ax.hist(np.clip(t_cal[pos] / n_seed[pos], 0, 3), bins=bins, color=tc.STAGE_COL["stage1"], alpha=0.7, label=f"Stage 1 (median {np.median(t_cal[pos]/n_seed[pos]):.2f})")
ax.hist(np.clip(t_star[pos] / n_seed[pos], 0, 3), bins=bins, histtype="step", lw=2, color="black", label=f"final (median {np.median(t_star[pos]/n_seed[pos]):.2f})")
ax.axvline(1, color=tc.COL["neutral"], ls=":"); ax.set_xlabel("$t_k / n_k$ (values >3 shown at 3)"); ax.set_ylabel("O-D pairs"); ax.legend()
ax.set_title(f"GLS adjustment factors: totals {n_seed.sum():,.0f} -> {t_cal.sum():,.0f} -> {t_star.sum():,.0f} trips/h")
tc.save_report(fig, "fig_S3_gls_adjustment")

# S4 — grade ablation (NB04, training stations)
ab = tc.DATA_PROCESSED / "ablation_grade_links.csv"
if ab.exists():
    d = pd.read_csv(ab); msk = d["train"].astype(bool).to_numpy() if "train" in d else train
    fig, axes = plt.subplots(1, 2, figsize=(10, 4.8), sharex=True, sharey=True)
    lim = 1.06 * max(d.v_obs[msk].max(), d.f_cal_grade[msk].max(), d.f_cal_flat[msk].max())
    for ax, c, ttl in zip(axes, ["f_cal_grade", "f_cal_flat"], ["grade-adjusted speed", "flat speed (ablation)"]):
        ax.scatter(d.v_obs[msk], d[c][msk], s=26, c=tc.COL["train"], edgecolor="white", lw=0.4)
        ax.plot([0, lim], [0, lim], color=tc.COL["neutral"], ls="--", lw=0.9); ax.set_xlim(0, lim); ax.set_ylim(0, lim)
        tc.metric_box(ax, tc.metrics(d[c][msk], d.v_obs[msk])); ax.set_title(f"Stage 1, {ttl}"); ax.set_xlabel("observed (trips/h)")
    axes[0].set_ylabel("modelled (trips/h)")
    tc.save_report(fig, "fig_S4_grade_ablation")

# S5 — train / held-out station map
fig, ax = plt.subplots(figsize=(8.5, 9))
basemap(ax); cordon_line(ax)
for msk, c, lab in [(train, tc.COL["train"], f"training (n={int(train.sum())})"), (test, tc.COL["test"], f"held-out (n={int(test.sum())})")]:
    ax.scatter(sx[msk], sy[msk], s=size(v_obs[msk]), c=c, edgecolor="black", lw=0.4, label=lab, zorder=5)
frame(ax); ax.legend(loc="lower left", frameon=True)
x0, x1 = ax.get_xlim(); y0, y1 = ax.get_ylim()
ax.annotate("N", xy=(x0 + (x1 - x0) * 0.95, y0 + (y1 - y0) * 0.90),
            xytext=(x0 + (x1 - x0) * 0.95, y0 + (y1 - y0) * 0.84),
            ha="center", va="center", fontsize=12, fontweight="bold",
            arrowprops=dict(arrowstyle="-|>", color="black", lw=1.6), zorder=10)
ax.set_title("Monitored stations by split (70/30, stratified on observed flow; size $\\propto$ flow)")
tc.save_report(fig, "fig_S5_split_map")

# S6 — k = 3 paths actually used for one pair (replaces the 'most distinct of 30 paths' picture)
from shapely.geometry import LineString

def _edgeset(p): return set(zip(p[:-1], p[1:]))
def _mean_overlap(ps):
    o = [len(_edgeset(ps[i]) & _edgeset(ps[j])) / max(len(_edgeset(ps[i]) | _edgeset(ps[j])), 1)
         for i in range(len(ps)) for j in range(i + 1, len(ps))]
    return float(np.mean(o)) if o else 1.0

def _path_segments(Gs, p):                       # real street geometry, falling back to a straight hop
    segs = []
    for a, b in zip(p[:-1], p[1:]):
        k = Gs[int(a)][int(b)].get("key")
        d = Gr.get_edge_data(int(a), int(b)) or {}
        d = d.get(k) or (next(iter(d.values())) if d else {})
        g = d.get("geometry")
        segs.append(np.asarray(g.coords) if g is not None else
                    np.array([[Gr.nodes[int(a)]["x"], Gr.nodes[int(a)]["y"]],
                              [Gr.nodes[int(b)]["x"], Gr.nodes[int(b)]["y"]]]))
    return segs

# candidate pairs: demand-ranked, 3 distinct paths, a real corridor, and moderate (not extreme) diversity
cands = []
for pid in np.argsort(-t_star):
    ri = int(rep_idx[pid])
    if ri < 0 or len(store_high[ri]) < 3:
        continue
    ps = store_high[ri][:3]
    if len({tuple(p) for p in ps}) < 3:                     # need three genuinely different paths
        continue
    Lm = sum(Gs_h[int(a)][int(b)]["len_m"] for a, b in zip(ps[0][:-1], ps[0][1:]))
    cands.append((pid, ps, Lm, _mean_overlap(ps), float(t_star[pid])))

# three alternatives that share a trunk but diverge: overlap in a mid band, corridor >= 1.2 km, high demand
good = [c for c in cands if c[2] >= 1200 and 0.25 <= c[3] <= 0.70]
pool = good if good else cands
pid, ps, Lm, ov, td = sorted(pool, key=lambda c: (-c[4], abs(c[3] - 0.5)))[0]

pcols   = [tc.COL["high"], tc.COL["train"], tc.COL["test"]]   # red / blue / orange
pstyles = ["-", "--", ":"]
pwid    = [4.6, 3.0, 1.8]                                     # staggered so shared trunks show all three
geoms   = [_path_segments(Gs_h, p) for p in ps]
allxy   = np.vstack([seg for g in geoms for seg in g])
xs, ys  = allxy[:, 0], allxy[:, 1]

fig, ax = plt.subplots(figsize=(7, 7.5)); pad = 0.12 * max(np.ptp(xs), np.ptp(ys)) + 120
ax.set_xlim(xs.min() - pad, xs.max() + pad); ax.set_ylim(ys.min() - pad, ys.max() + pad)
basemap(ax, lw=0.5)
for i, (p, segs, c, ls, lw_) in enumerate(zip(ps, geoms, pcols, pstyles, pwid)):
    cost = sum(Gs_h[int(a)][int(b)]["cost_high"] for a, b in zip(p[:-1], p[1:]))
    for s_i, seg in enumerate(segs):
        ax.plot(seg[:, 0], seg[:, 1], color=c, lw=lw_, ls=ls, alpha=0.9, solid_capstyle="round",
                zorder=3 + i, label=(f"path {i+1}: {cost:.1f} min (1/3 of demand)" if s_i == 0 else None))
# origin / destination markers
for lab, n in (("O", ps[0][0]), ("D", ps[0][-1])):
    ax.scatter(Gr.nodes[int(n)]["x"], Gr.nodes[int(n)]["y"], s=55, c="black", zorder=8)
    ax.annotate(lab, (Gr.nodes[int(n)]["x"], Gr.nodes[int(n)]["y"]), xytext=(6, 6),
                textcoords="offset points", fontsize=11, fontweight="bold", zorder=9)
ax.legend(loc="best", frameon=True); ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
x0, x1 = ax.get_xlim(); y0, y1 = ax.get_ylim()
ax.annotate("N", xy=(x0 + (x1 - x0) * 0.95, y0 + (y1 - y0) * 0.90),
            xytext=(x0 + (x1 - x0) * 0.95, y0 + (y1 - y0) * 0.84),
            ha="center", va="center", fontsize=12, fontweight="bold",
            arrowprops=dict(arrowstyle="-|>", color="black", lw=1.6), zorder=10)
ax.set_title("The $k$=3 shortest paths that share one O-D pair's demand")
tc.save_report(fig, "fig_S6_k_paths_example")
print(f"S6 pair {pid}: corridor {Lm/1000:.1f} km, mean path overlap {ov:.2f}, demand {td:.1f} trips/h")

# S7 — three standalone input figures

# (1) grade-adjusted speed curve
g = np.linspace(-10, 10, 401)
fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(g, np.maximum(tc.VMIN, tc.VFLAT - tc.KUP * np.clip(g, 0, None) + tc.KDOWN * np.clip(-g, 0, None)), color="black", lw=2)
ax.axhline(tc.VFLAT, color=tc.COL["neutral"], ls=":")
ax.set_xlabel("grade (%)"); ax.set_ylabel("speed (km/h)")
ax.set_title("Grade-adjusted speed (Parkin & Rotheram 2010)")
tc.save_report(fig, "fig_S7a_speed")

# (2) grade distribution on the routable network
fig, ax = plt.subplots(figsize=(5, 4))
ax.hist(edges["grade"], bins=np.arange(-16, 16.5, 0.5), color=tc.COL["neutral"]); ax.set_yscale("log")
ax.set_xlabel("signed grade (%)"); ax.set_ylabel("directed edges (log)")
ax.set_title("Grade on the routable network")
tc.save_report(fig, "fig_S7b_grade")

# (3) observed count targets
fig, ax = plt.subplots(figsize=(5, 4))
ax.hist(v_obs, bins=20, color=tc.COL["obs"]); ax.axvline(np.median(v_obs), color="white", ls="--")
ax.set_xlabel("observed peak-hour flow (trips/h)"); ax.set_ylabel("stations")
ax.set_title(f"Count targets (n={L}, median {np.median(v_obs):.0f})")
tc.save_report(fig, "fig_S7c_counts")

# S8 — LTS map
fig, ax = plt.subplots(figsize=(8.5, 9))
for l, lw_ in [(4, 0.6), (3, 0.6), (2, 0.7), (1, 1.1)]:
    edges[edges.lts == l].plot(ax=ax, color=tc.LTS_COL[l], lw=lw_, zorder=l)
cordon_line(ax); frame(ax)
km = edges.drop_duplicates(subset=["u", "v"]).groupby("lts").len_m.sum() / 1000
ax.legend(handles=[Line2D([], [], color=tc.LTS_COL[l], lw=3, label=f"LTS {l} ({100*km.get(l,0)/km.sum():.0f}% of km)") for l in [1, 2, 3, 4]],
          loc="lower left", frameon=True)
x0, x1 = ax.get_xlim(); y0, y1 = ax.get_ylim()
ax.annotate("N", xy=(x0 + (x1 - x0) * 0.95, y0 + (y1 - y0) * 0.90),
            xytext=(x0 + (x1 - x0) * 0.95, y0 + (y1 - y0) * 0.84),
            ha="center", va="center", fontsize=12, fontweight="bold",
            arrowprops=dict(arrowstyle="-|>", color="black", lw=1.6), zorder=10)
ax.set_title("Level of Traffic Stress on the routable network")
tc.save_report(fig, "fig_S8_lts_map")

  saved report figure  fig_S1_component_ladder.pdf/.png
  saved report figure  fig_S2_demand_by_movement.pdf/.png
  saved report figure  fig_S3_gls_adjustment.pdf/.png
  saved report figure  fig_S4_grade_ablation.pdf/.png
  saved report figure  fig_S5_split_map.pdf/.png
  saved report figure  fig_S6_k_paths_example.pdf/.png
S6 pair 69: corridor 2.2 km, mean path overlap 0.58, demand 25.1 trips/h
  saved report figure  fig_S7a_speed.pdf/.png
  saved report figure  fig_S7b_grade.pdf/.png
  saved report figure  fig_S7c_counts.pdf/.png
  saved report figure  fig_S8_lts_map.pdf/.png


## Numbers for the write-up

Emits the headline numbers quoted in the text (fit metrics, demand totals, share, split, detours, coverage)
to the console and to `results_numbers.txt` — part of this notebook's end-of-run key-results output.


In [13]:
# ══════════════════════════════════════════════════════════════════════
# FIGURE — Study-area GIS overview + inbound-load entry points (merged)
#   S10 base layers (network · SA1/DZN zones · cordon · count stations)
#   with cordon entry points drawn as proportional symbols scaled by
#   inbound-external trip load and labelled by entry ID.
#   Self-contained for the GIS layers; the load overlay uses od_snapped
#   (falls back to uniform markers if it isn't in scope).
#   Fonts come from tc.set_style() — do NOT set rcParams here.
# ══════════════════════════════════════════════════════════════════════
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

print("\n" + "=" * 60); print("FIGURE — Study area + inbound-load entries"); print("=" * 60)

# ── Network → project → G_scc ───────────────────────────────────────────
GRAPHML = tc.NETWORK_GRAPHML if tc.NETWORK_GRAPHML.exists() else next(
    (p for p in sorted(tc.DATA_RAW.glob("osm_network_*.graphml"))[::-1]), None)
assert GRAPHML is not None, f"No network graphml found in {tc.DATA_PROCESSED} / {tc.DATA_RAW}"
Gp = ox.project_graph(ox.load_graphml(str(GRAPHML)), to_crs=tc.PROJECTED_CRS)
scc_nodes = max(nx.strongly_connected_components(Gp), key=len)
G_scc = Gp.subgraph(scc_nodes).copy()
node_x = nx.get_node_attributes(Gp, "x"); node_y = nx.get_node_attributes(Gp, "y")
print(f"  Network {GRAPHML.name}: {len(Gp.nodes):,} → G_scc {len(scc_nodes):,} nodes")

# ── Count stations → snap to G_scc ──────────────────────────────────────
sites = pd.read_csv(tc.DATA_RAW / "bicycle_count_sites.csv")
sites.columns = [c.strip() for c in sites.columns]
sites = sites.dropna(subset=["lat", "lon"])
site_pts = gpd.GeoSeries(gpd.points_from_xy(sites.lon.astype(float), sites.lat.astype(float)),
                         crs=tc.LONLAT_CRS).to_crs(tc.PROJECTED_CRS)
site_nodes, _ = ox.distance.nearest_nodes(G_scc, X=site_pts.x.values, Y=site_pts.y.values, return_dist=True)
snap_x = np.array([node_x[n] for n in site_nodes]); snap_y = np.array([node_y[n] for n in site_nodes])

# ── Entry points → snap to G_scc, keep entry IDs ────────────────────────
entries = pd.read_csv(tc.DATA_RAW / "cordon_entry_points.csv", dtype=str).apply(lambda c: c.str.strip())
id_col = next((c for c in entries.columns if c.lower() in
               ("entry_id", "id", "entry", "entryid", "name")), entries.columns[0])
entry_pts = gpd.GeoSeries(gpd.points_from_xy(entries.x.astype(float), entries.y.astype(float)),
                          crs=tc.LONLAT_CRS).to_crs(tc.PROJECTED_CRS)
entry_nodes, _ = ox.distance.nearest_nodes(G_scc, X=entry_pts.x.values, Y=entry_pts.y.values, return_dist=True)
entry_x = np.array([node_x[n] for n in entry_nodes]); entry_y = np.array([node_y[n] for n in entry_nodes])

# ── Inbound-external load per entry (from od_snapped; graceful fallback) ─
try:
    eset = set(entries[id_col])
    _load = (od_snapped[(od_snapped.movement == "inbound_external")
                        & od_snapped.origin_point.isin(eset)]
             .groupby("origin_point").trips.sum())
    entry_load = entries[id_col].map(_load).fillna(0).to_numpy(dtype=float)
    have_load = entry_load.sum() > 0
    print(f"  Entry load: {int(entry_load.sum()):,} inbound-external trips over {len(entries)} entries")
except (NameError, KeyError) as e:
    entry_load = np.zeros(len(entries)); have_load = False
    print(f"  Entry load unavailable ({type(e).__name__}) — uniform markers")

# proportional symbols: scatter `s` is area in pt², so area ∝ load
S_MIN, S_MAX = 50.0, 900.0
lmax = entry_load.max() if have_load else 0.0
entry_s = (S_MIN + (entry_load / lmax) * (S_MAX - S_MIN)) if have_load else np.full(len(entries), 110.0)

# ── Zones: SA1 + DZN polygons ───────────────────────────────────────────
sa1_codes = set(pd.read_csv(tc.DATA_RAW / "SA1s_in_Boundary_Cordon.csv", dtype=str)
                .apply(lambda c: c.str.strip())["SA1_7DIG16"])
SA1_ZIP = tc.DATA_RAW / "1270055001_sa1_2016_aust_shape.zip"
if SA1_ZIP.exists():
    _s = gpd.read_file(f"zip://{SA1_ZIP}")
    _c = next((c for c in _s.columns if "SA1" in c.upper() and "DIG" in c.upper()),
              next(c for c in _s.columns if "SA1" in c.upper()))
    _s[_c] = _s[_c].astype(str).str.strip()
    sa1_poly = _s[_s[_c].isin(set(map(str, sa1_codes)))].to_crs(tc.PROJECTED_CRS)
else:
    _ll = pd.read_csv(tc.DATA_RAW / "SA1s_in_Boundary_Cordon.csv")
    _ll = _ll[_ll["SA1_7DIG16"].astype(str).isin(set(map(str, sa1_codes)))]
    sa1_poly = gpd.GeoDataFrame(geometry=gpd.points_from_xy(_ll["X"], _ll["Y"]),
                                crs=tc.LONLAT_CRS).to_crs(tc.PROJECTED_CRS)
    sa1_poly["geometry"] = sa1_poly.buffer(120)

dzn_inside = set(pd.read_csv(tc.DATA_RAW / "DZNs_in_Boundary_Cordon.csv", dtype=str)
                 .apply(lambda c: c.str.strip())["DZN_CODE_2016"])
dzn_all = gpd.read_file(str(tc.DATA_RAW / "80000_dzn_2016_aust_geopackage" / "ASGS 2016 DZN.gpkg"))
_dc = ([c for c in dzn_all.columns if "DZN" in c.upper() and "CODE" in c.upper()]
       or [c for c in dzn_all.columns if "DZN" in c.upper()])
dzn_all["dzn_code"] = dzn_all[_dc[0]].astype(str).str.strip()
dzn_p = dzn_all[dzn_all["dzn_code"].isin(dzn_inside)].to_crs(tc.PROJECTED_CRS)

# ── Cordon outline + network edges ──────────────────────────────────────
cordon_bnd = gpd.read_file(tc.STUDY_AREA_CORDON).to_crs(tc.PROJECTED_CRS).boundary
edges_scc = ox.graph_to_gdfs(G_scc, nodes=False, edges=True)

# ── Draw ────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 11))
ax.set_facecolor("white"); fig.patch.set_facecolor("white")
edges_scc.plot(ax=ax, color="#cfcfcf", linewidth=0.35, zorder=1)
dzn_p.plot(ax=ax, facecolor="#3b6fb0", alpha=0.18, edgecolor="#2f5986", linewidth=0.4, zorder=2)
sa1_poly.plot(ax=ax, facecolor="#4caf7d", alpha=0.18, edgecolor="#2f7d55", linewidth=0.4, zorder=3)
cordon_bnd.plot(ax=ax, color="#b00020", linewidth=2.2, zorder=6)

# count stations (small dots)
ax.scatter(snap_x, snap_y, s=28, marker="o", facecolor="#111111",
           edgecolor="white", linewidth=0.5, zorder=7)

# entry points: proportional triangles scaled by inbound load, labelled
ax.scatter(entry_x, entry_y, s=entry_s, marker="o", facecolor="#f0a500",
           edgecolor="black", linewidth=0.7, alpha=0.9, zorder=8)
for xi, yi, lab in zip(entry_x, entry_y, entries[id_col]):
    ax.annotate(str(lab), (xi, yi), fontsize=7, ha="center", va="bottom",
                xytext=(0, 7), textcoords="offset points",
                color="#7a2d00", fontweight="bold", zorder=9)

# ── Category legend ─────────────────────────────────────────────────────
cat_handles = [
    Line2D([0], [0], color="#b00020", lw=2.2, label=f"Study-area cordon ({len(sa1_codes)} SA1)"),
    Patch(facecolor="#4caf7d", alpha=0.35, edgecolor="#2f7d55", label="SA1 zones"),
    Patch(facecolor="#3b6fb0", alpha=0.35, edgecolor="#2f5986", label="DZN zones"),
    Line2D([0], [0], marker="o", color="none", markerfacecolor="#111111", markeredgecolor="white",
           markersize=7, label=f"Count stations (n={len(sites)})"),
    Line2D([0], [0], marker="o", color="none", markerfacecolor="#f0a500", markeredgecolor="black",
           markersize=10, label=f"Entry points (n={len(entries)})"),
]
leg1 = ax.legend(handles=cat_handles, loc="upper left", frameon=True,
                 framealpha=0.95, fontsize=10, borderpad=0.8, title="Layers")
leg1.get_title().set_fontweight("bold"); ax.add_artist(leg1)


# scale bar (1 km) + north arrow
x0, x1 = ax.get_xlim(); y0, y1 = ax.get_ylim()
bx, by = x0 + (x1 - x0) * 0.06, y0 + (y1 - y0) * 0.05
ax.plot([bx, bx + 1000], [by, by], color="black", lw=2.5, zorder=10)
ax.text(bx + 500, by + (y1 - y0) * 0.012, "1 km", ha="center", va="bottom", fontsize=9)
ax.annotate("N", xy=(x0 + (x1 - x0) * 0.95, y0 + (y1 - y0) * 0.90),
            xytext=(x0 + (x1 - x0) * 0.95, y0 + (y1 - y0) * 0.84),
            ha="center", va="center", fontsize=12, fontweight="bold",
            arrowprops=dict(arrowstyle="-|>", color="black", lw=1.6))

ax.set_title("City of Sydney study area: zones, count network and cordon entry loads",
             fontsize=13, pad=12)
ax.set_xlabel("Easting (m, MGA Zone 56)"); ax.set_ylabel("Northing (m, MGA Zone 56)")
ax.set_aspect("equal"); ax.ticklabel_format(style="plain", axis="both")
fig.tight_layout()
tc.save_report(fig, "fig_S11_study_area_entry_loads")


FIGURE — Study area + inbound-load entries
  Network network_lts_grade.graphml: 21,691 → G_scc 21,212 nodes
  Entry load: 2,462 inbound-external trips over 30 entries
  saved report figure  fig_S11_study_area_entry_loads.pdf/.png


In [14]:
# ------------------------------------------------------------------
# NB06 — DEM grade artefact on bridges (evidence for the Discussion)
# Samples the EXISTING ELVIS DEM raster directly at node coordinates, so the
# raw (un-zeroed) grade is recovered without re-running NB01.
# ------------------------------------------------------------------
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import rasterio
from rasterio.warp import transform as rio_transform
tc.set_style()

DEM = tc.DATA_PROCESSED / "dem_metric.tif"
if not DEM.exists():
    DEM = tc.DATA_RAW / "sydney_elvis_dem.tif"
assert DEM.exists(), f"DEM raster not found (looked for dem_metric.tif / sydney_elvis_dem.tif in {tc.DATA_PROCESSED} and {tc.DATA_RAW})"

Gr = tc.load_network()
def _s(x):
    x = x[0] if isinstance(x, list) else x
    return str(x) if x is not None else ""

# --- sample node elevations straight from the DEM -----------------
nodes = list(Gr.nodes)
xs = np.array([Gr.nodes[n]["x"] for n in nodes], float)
ys = np.array([Gr.nodes[n]["y"] for n in nodes], float)
with rasterio.open(DEM) as src:
    if src.crs and str(src.crs) != str(tc.PROJECTED_CRS):
        sx, sy = rio_transform(tc.PROJECTED_CRS, src.crs, xs.tolist(), ys.tolist())
    else:
        sx, sy = xs, ys
    samp = np.array([val[0] for val in src.sample(np.c_[sx, sy])], float)
    if src.nodata is not None:
        samp[samp == src.nodata] = np.nan
elev = dict(zip(nodes, samp))
print(f"  sampled {np.isfinite(samp).sum():,}/{len(nodes):,} node elevations from {DEM.name}")

# --- raw per-edge grade from node elevations ----------------------
rows = []
for u, v, k, d in Gr.edges(keys=True, data=True):
    eu, ev = elev.get(u, np.nan), elev.get(v, np.nan)
    L = float(d.get("len_m", 0.0))
    if not (np.isfinite(eu) and np.isfinite(ev)) or L <= 0:
        continue
    raw = 100.0 * (ev - eu) / L                                  # raw DEM grade (%), un-zeroed
    brg = _s(d.get("bridge")).lower() in {"yes","viaduct","boardwalk","cantilever","movable"}
    rows.append((abs(raw), brg, _s(d.get("highway")).lower(), L/1000.0, _s(d.get("name"))))

df  = pd.DataFrame(rows, columns=["raw_abs","is_bridge","highway","km","name"])
br  = df.is_bridge.to_numpy()
ref = (~df.is_bridge & ~df.highway.isin(["steps","corridor"])).to_numpy()
raw, km = df.raw_abs.to_numpy(), df.km.to_numpy()

VFLAT, KUP, VMIN = tc.VFLAT, tc.KUP, tc.VMIN                      # 21.6, 1.44, 6.0
FLOOR_G   = (VFLAT - VMIN) / KUP                                  # ~10.8%: uphill speed pinned at v_min
PLAUSIBLE = 5.0        # Austroads (2017) GRD Part 6A: desirable max cycle-path grade, 5% (1:20)
pc   = lambda a, q: (np.percentile(a[np.isfinite(a)], q) if np.isfinite(a).any() else np.nan)
infl = VFLAT / np.maximum(VMIN, VFLAT - KUP*raw[br]) - 1.0

print("[08-diag] DEM grade artefact on bridges")
print(f"  bridge edges: {int(br.sum()):,} ({km[br].sum():.1f} km)")
print(f"  raw |grade| bridges  : p50 {pc(raw[br],50):.2f}%  p90 {pc(raw[br],90):.2f}%  "
      f"p95 {pc(raw[br],95):.2f}%  max {np.nanmax(raw[br]):.2f}%")
print(f"  raw |grade| at-grade : p50 {pc(raw[ref],50):.2f}%  p90 {pc(raw[ref],90):.2f}%  p95 {pc(raw[ref],95):.2f}%")
impl = br & (raw > PLAUSIBLE)
print(f"  bridges with raw |grade| > {PLAUSIBLE:.0f}% (Austroads max): {int(impl.sum()):,} "
      f"({km[impl].sum():.1f} km, {100*impl.sum()/max(br.sum(),1):.0f}% of bridge edges)")
print(f"  implied uphill time inflation if not zeroed: median +{100*np.nanmedian(infl):.0f}%  "
      f"max +{100*np.nanmax(infl):.0f}%")
print(f"  bridge edges hitting the {VMIN:.0f} km/h floor (|grade|>{FLOOR_G:.1f}%): {int((br & (raw>FLOOR_G)).sum()):,}")
for key in ["harbour", "anzac"]:
    m = br & df.name.str.lower().str.contains(key, na=False).to_numpy()
    if m.sum():
        print(f"  {key:>8} crossing: raw |grade| up to {np.nanmax(raw[m]):.1f}% over {km[m].sum():.2f} km")

df.to_csv(tc.DATA_PROCESSED / "dem_bridge_grade_artefact.csv", index=False)
fig, ax = plt.subplots(figsize=(6.5, 3.6))
bins = np.linspace(0, max(16.0, np.nanmax(raw[br])), 40)
x0, x1 = ax.get_xlim(); y0, y1 = ax.get_ylim()
ax.annotate("N", xy=(x0 + (x1 - x0) * 0.95, y0 + (y1 - y0) * 0.90),
            xytext=(x0 + (x1 - x0) * 0.95, y0 + (y1 - y0) * 0.84),
            ha="center", va="center", fontsize=12, fontweight="bold",
            arrowprops=dict(arrowstyle="-|>", color="black", lw=1.6), zorder=10)
ax.hist(raw[ref], bins=bins, density=True, color=tc.COL["neutral"], alpha=0.8, label="at-grade edges")
ax.hist(raw[br],  bins=bins, density=True, histtype="step", lw=1.8, color=tc.COL["high"], label="bridge edges")
ax.axvline(PLAUSIBLE, color="black", ls="--", lw=1); ax.text(PLAUSIBLE, ax.get_ylim()[1]*0.90, " Austroads 5%", fontsize=8)
ax.axvline(FLOOR_G,   color="black", ls=":",  lw=1); ax.text(FLOOR_G,   ax.get_ylim()[1]*0.74, f" {VMIN:.0f} km/h floor", fontsize=8)
ax.set_xlabel("raw DEM |grade| (%), before structure-zeroing"); ax.set_ylabel("density")
ax.set_title("DEM over-states gradient on bridges"); ax.legend()
tc.save_report(fig, "fig_dem_bridge_grade_artefact")

  sampled 21,212/21,212 node elevations from sydney_elvis_dem.tif
[08-diag] DEM grade artefact on bridges
  bridge edges: 477 (47.9 km)
  raw |grade| bridges  : p50 1.51%  p90 10.75%  p95 28.92%  max 92.36%
  raw |grade| at-grade : p50 1.86%  p90 6.23%  p95 8.04%
  bridges with raw |grade| > 5% (Austroads max): 111 (5.5 km, 23% of bridge edges)
  implied uphill time inflation if not zeroed: median +11%  max +260%
  bridge edges hitting the 6 km/h floor (|grade|>10.8%): 48
   harbour crossing: raw |grade| up to 60.2% over 0.01 km
     anzac crossing: raw |grade| up to 7.5% over 1.84 km
  saved report figure  fig_dem_bridge_grade_artefact.pdf/.png


In [15]:
# ============================================================
# NB06 — classify top-error junction stations: re-snap vs behavioural
# Rebuilds calibrated full-network edge flows from the cached path stores,
# then for each flagged site compares the counter's matched edge with the
# heaviest modelled-flow edge within SEARCH_M and returns a verdict.
# ============================================================
import numpy as np, pandas as pd
import thesis_common as tc

SITES_TO_CHECK = [74, 66, 45, 96]   # first-mechanism (junction-leg) top-error sites
SEARCH_M       = 250.0              # same radius as the Results corridor diagnostic
NODE_NEAR_M    = 35.0               # "same junction" if a candidate endpoint is this close to the counter junction

Gr    = tc.load_network()
cal   = pd.read_csv(tc.DATA_PROCESSED / "od_final_t.csv").sort_values("pair_idx").reset_index(drop=True)
snap  = pd.read_csv(tc.DATA_PROCESSED / "count_link_snap.csv")
fit   = pd.read_csv(tc.DATA_PROCESSED / "stage3_link_fit.csv")
final = tc.read_final(); pi = float(final["pi_star"])
t_star = (cal["t_final"] if "t_final" in cal else cal["t"]).to_numpy(float)

# --- calibrated full-network edge flows (same construction as the R5 flow map) ---
exp_pairs, assembly, _ = tc.expand_pairs(cal)
tc.set_costs(Gr, lam=tc.STRESS_PENALTY, beta=0.0, flat=False)
store_high = tc.route_store(tc.to_simple(Gr, "cost_high"), exp_pairs, tc.K_STORE, "cost_high", tc.store_tag("high"), n_jobs=1)
store_low  = tc.route_store(tc.to_simple(Gr, "cost_low"),  exp_pairs, tc.K_STORE, "cost_low",  tc.store_tag("low"),  n_jobs=1)
edge_flow = {}
for wts, store in [(pi * t_star, store_low), ((1 - pi) * t_star, store_high)]:
    for e, fv in tc.edge_flows_spread(store, tc.K_PATHS, wts, assembly).items():
        edge_flow[e] = edge_flow.get(e, 0.0) + fv

# --- geometry helpers (Gr is projected to metres) ---
X = {n: (Gr.nodes[n]["x"], Gr.nodes[n]["y"]) for n in Gr.nodes}
def _mid(u, v):  return ((X[u][0] + X[v][0]) / 2, (X[u][1] + X[v][1]) / 2)
def _brg(u, v):  return np.degrees(np.arctan2(X[v][0] - X[u][0], X[v][1] - X[u][1])) % 360
def _name(u, v):
    d = Gr.get_edge_data(u, v)
    return tc._name(list(d.values())[0]) if d else None
def _norm(s):    return tc.abbreviate(str(s)).strip().lower() if s else ""

# --- all edges as a lookup table (midpoint, name, bearing, flow) ---
rows = []
for u, v, k, d in Gr.edges(keys=True, data=True):
    mx, my = _mid(u, v)
    rows.append((u, v, mx, my, _name(u, v), _brg(u, v), edge_flow.get((u, v), 0.0)))
E = pd.DataFrame(rows, columns=["u", "v", "mx", "my", "name", "brg", "flow"])

# --- classify each site ---
fitm = fit.set_index("SiteID")
print(f"edge-flow classifier  (pi*={pi:.2f}, search {SEARCH_M:.0f} m)\n" + "=" * 70)
for site in SITES_TO_CHECK:
    s = snap[snap.SiteID == site]
    if s.empty:
        print(f"Site {site}: not in count_link_snap.csv"); continue
    su, sv = int(s.iloc[0]["u"]), int(s.iloc[0]["v"])
    cmx, cmy = _mid(su, sv); cname = _name(su, sv)
    matched = edge_flow.get((su, sv), 0.0) + edge_flow.get((sv, su), 0.0)
    obs = fitm.loc[site, "v_obs"] if site in fitm.index else np.nan
    mod = fitm.loc[site, "f_final"] if site in fitm.index else np.nan
    loc = fitm.loc[site, "location"] if site in fitm.index else ""

    near = E[(np.hypot(E.mx - cmx, E.my - cmy) <= SEARCH_M) &
             ~((E.u == su) & (E.v == sv)) & ~((E.u == sv) & (E.v == su))].copy()
    near = near[near.flow > 0].sort_values("flow", ascending=False)
    print(f"\nSite {site}  {loc}")
    print(f"  counter edge '{cname}'  obs {obs:.0f} / modelled {mod:.0f}  (flow on matched edge {matched:.0f})")
    if near.empty:
        print("  no nearby modelled flow -> NOT a matching artefact (check cordon/demand)"); continue

    h = near.iloc[0]
    dist = float(np.hypot(h.mx - cmx, h.my - cmy))
    node_d = min(np.hypot(X[n][0] - X[j][0], X[n][1] - X[j][1])
                 for n in (int(h.u), int(h.v)) for j in (su, sv))
    incident = {int(h.u), int(h.v)} & {su, sv}
    same_junction = bool(incident) or node_d <= NODE_NEAR_M

    if _norm(h["name"]) == _norm(cname) and cname:
        verdict = "RE-SNAP  (same street, adjacent segment)"
    elif same_junction:
        verdict = "CHECK AERIAL  (same junction, different leg -> re-snap only if the counter physically sits on this leg)"
    else:
        verdict = f"BEHAVIOURAL  (separate road ~{dist:.0f} m away -> do not re-snap)"
    print(f"  heaviest nearby: '{h['name']}'  flow {h.flow:.0f}  dist {dist:.0f} m  "
          f"junction-adjacent={same_junction}")
    print(f"  VERDICT: {verdict}")
    for _, r in near.head(3).iterrows():
        print(f"      candidate '{r['name']}'  flow {r.flow:.0f}  "
              f"dist {np.hypot(r.mx-cmx, r.my-cmy):.0f} m  brg {r.brg:.0f}")

  [high_beta0_grade_k5_sprd5] loaded cached path store (8,721 pairs, k=5)
  [low_lam2.4_beta0_grade_k5_sprd5] loaded cached path store (8,721 pairs, k=5)
edge-flow classifier  (pi*=0.64, search 250 m)

Site 74  Moore Park Rd / Lang Rd
  counter edge 'Moore Park Road'  obs 377 / modelled 4  (flow on matched edge 4)
  heaviest nearby: 'Lang Road'  flow 278  dist 26 m  junction-adjacent=True
  VERDICT: CHECK AERIAL  (same junction, different leg -> re-snap only if the counter physically sits on this leg)
      candidate 'Lang Road'  flow 278  dist 26 m  brg 182
      candidate 'Moore Park Road'  flow 273  dist 114 m  brg 269
      candidate 'Moore Park Road'  flow 273  dist 95 m  brg 267

Site 66  Cleveland St / Anzac Pde
  counter edge 'Cleveland Street'  obs 315 / modelled 80  (flow on matched edge 7)
  heaviest nearby: 'Anzac Parade'  flow 205  dist 8 m  junction-adjacent=True
  VERDICT: CHECK AERIAL  (same junction, different leg -> re-snap only if the counter physically sits on this 

In [16]:
# ============================================================
# NB06 — cordon entry point vs count-station alignment
# For each of the 30 entry points, finds a nearby count station and checks
# whether the entry's snapped node lies on that station's matched edge. If
# not, external (inbound_external) demand enters off the counted edge and
# the counter reads low through no fault of the demand model.
# ============================================================
import numpy as np, pandas as pd
import thesis_common as tc

SEARCH_M    = 150.0    # treat an entry point as "on" a count station if within this
NODE_NEAR_M = 20.0     # entry node counts as on the station edge if within this of an endpoint

Gr   = tc.load_network()
cal  = pd.read_csv(tc.DATA_PROCESSED / "od_calibrated_t.csv")
snap = pd.read_csv(tc.DATA_PROCESSED / "count_link_snap.csv")
fit  = pd.read_csv(tc.DATA_PROCESSED / "stage3_link_fit.csv").set_index("SiteID")

try:
    ep = pd.read_csv(tc.DATA_RAW / "cordon_entry_points.csv")
    entry_name = {str(r.entry_id): getattr(r, "road_name", str(r.entry_id)) for r in ep.itertuples()}
except Exception:
    entry_name = {}

X = {n: (Gr.nodes[n]["x"], Gr.nodes[n]["y"]) for n in Gr.nodes}
def _name(u, v):
    d = Gr.get_edge_data(u, v); return tc._name(list(d.values())[0]) if d else None

# entry_id -> snapped node (external origins)
ext = cal[cal.movement == "inbound_external"].dropna(subset=["o_node"]).drop_duplicates("o")
entry_node = {str(r.o): int(r.o_node) for r in ext.itertuples()}

# count stations -> matched-edge nodes + midpoint
rows = []
for r in snap.itertuples():
    u, v = int(r.u), int(r.v)
    if u in X and v in X:
        rows.append((int(r.SiteID), u, v, (X[u][0]+X[v][0])/2, (X[u][1]+X[v][1])/2))
stn = pd.DataFrame(rows, columns=["SiteID", "u", "v", "mx", "my"])

print(f"cordon entry-point vs count alignment  (search {SEARCH_M:.0f} m)\n" + "=" * 72)
flagged = []
for eid, en in entry_node.items():
    ex, ey = X[en]
    stn["d"] = np.hypot(stn.mx - ex, stn.my - ey)
    near = stn[stn.d <= SEARCH_M].sort_values("d")
    if near.empty:
        continue
    s = near.iloc[0]
    node_gap = min(np.hypot(ex - X[n][0], ey - X[n][1]) for n in (int(s.u), int(s.v)))
    on_edge  = (en in (int(s.u), int(s.v))) or node_gap <= NODE_NEAR_M
    obs = fit.loc[int(s.SiteID), "v_obs"]   if int(s.SiteID) in fit.index else np.nan
    mod = fit.loc[int(s.SiteID), "f_final"] if int(s.SiteID) in fit.index else np.nan
    status = "ALIGNED" if on_edge else "MISALIGNED -> re-snap entry node onto the station edge"
    print(f"entry {entry_name.get(eid, eid):<22} ~ Site {int(s.SiteID):>3} "
          f"'{_name(int(s.u), int(s.v))}'  dist {s.d:4.0f} m  node-gap {node_gap:4.0f} m  "
          f"obs {obs:5.0f}/mod {mod:5.0f}  -> {status}")
    if not on_edge:
        flagged.append((entry_name.get(eid, eid), int(s.SiteID), round(s.d), round(node_gap)))

print("\nSummary:")
if flagged:
    print(f"  {len(flagged)} entry point(s) sit near a counter but off its edge:")
    for name, site, d, nd in flagged:
        print(f"    {name} -> Site {site}: node gap {nd} m (external demand enters off the counted edge)")
else:
    print("  every entry point near a counter is already on the counter's edge.")

cordon entry-point vs count alignment  (search 150 m)
entry New South Head Road    ~ Site  37 'Neild Avenue'  dist    7 m  node-gap    0 m  obs    19/mod    10  -> ALIGNED
entry Oxford Street          ~ Site  74 'Moore Park Road'  dist   23 m  node-gap    0 m  obs   377/mod     4  -> ALIGNED
entry Alison Road/Anzac Parade ~ Site  67 'Alison Road'  dist   75 m  node-gap   75 m  obs   192/mod   219  -> MISALIGNED -> re-snap entry node onto the station edge
entry Todman Avenue          ~ Site 100 'South Dowling Street'  dist   21 m  node-gap   21 m  obs    48/mod     3  -> MISALIGNED -> re-snap entry node onto the station edge
entry Dalmeny Avenue         ~ Site  85 'Dalmeny Avenue'  dist    4 m  node-gap    0 m  obs    28/mod    36  -> ALIGNED
entry Bourke Road            ~ Site  26 'Bourke Road'  dist   16 m  node-gap    9 m  obs    94/mod   105  -> ALIGNED
entry Sydney Harbour Bridge  ~ Site  21 'Harbour Bridge Cycleway'  dist    2 m  node-gap    0 m  obs   223/mod   292  -> ALIGNED

S

In [17]:
import osmnx as ox, numpy as np, pandas as pd
import matplotlib.pyplot as plt
import thesis_common as tc

tc.set_style()
Gr   = tc.load_network()
snap = pd.read_csv(tc.DATA_PROCESSED / "count_link_snap.csv")
X = {n: (Gr.nodes[n]["x"], Gr.nodes[n]["y"]) for n in Gr.nodes}

sites = snap[snap.SiteID.isin([65, 66, 74])]
mids  = [((X[int(r.u)][0]+X[int(r.v)][0])/2, (X[int(r.u)][1]+X[int(r.v)][1])/2) for r in sites.itertuples()]
cx, cy = np.mean([m[0] for m in mids]), np.mean([m[1] for m in mids])

edges = ox.graph_to_gdfs(Gr, nodes=False).to_crs(tc.PROJECTED_CRS)
box = edges.cx[cx-700:cx+700, cy-700:cy+700].copy()

def _s(x):
    x = x[0] if isinstance(x, list) else x
    return "" if x is None else str(x)
box["name_s"] = box["name"].apply(_s) if "name" in box.columns else ""
box["hw_s"]   = box["highway"].apply(_s)
box["lts_i"]  = box["lts"].apply(lambda v: int(v[0]) if isinstance(v, list) else int(v))

print("edges near Moore Park by LTS:")
print(box.groupby("lts_i").size())
print("\nnamed through-paths (cycleway / path / footway):")
paths = box[box.hw_s.str.contains("cycleway|path|footway", na=False)]
print(paths[["name_s", "hw_s", "lts_i"]].drop_duplicates().head(30).to_string(index=False))

# --- figure: create fig+ax explicitly so save_report gets the figure ---
fig, ax = plt.subplots(figsize=(8, 8))
for lts, sub in box.groupby("lts_i"):
    sub.plot(ax=ax, color=tc.LTS_COL.get(int(lts), "#999999"), linewidth=0.8, label=f"LTS {int(lts)}")
for r, (mx, my) in zip(sites.itertuples(), mids):
    ax.annotate(f"Site {int(r.SiteID)}", (mx, my), fontsize=9, weight="bold", ha="center", zorder=5,
                bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.8))
ax.set_aspect("equal"); ax.set_axis_off()
ax.legend(title="Level of Traffic Stress", fontsize=8, loc="upper right")
ax.set_title("Network around Moore Park (LTS)")
tc.save_report(fig, "fig_moorepark_lts")

edges near Moore Park by LTS:
lts_i
1    760
2    460
3    315
4     35
dtype: int64

named through-paths (cycleway / path / footway):
                                            name_s     hw_s  lts_i
                                    Federation Way cycleway      1
                                               nan cycleway      3
                                               nan cycleway      1
                          Anzac Parade Shared Path cycleway      1
                        Albert Tibby Cotter Bridge cycleway      1
                                               nan  footway      1
Moore Park Road and Fitzroy Street Pop-Up Cycleway cycleway      3
                                               nan     path      1
                          Anzac Parade Shared Path cycleway      3
                                Peter Finch Avenue  footway      1
                                     Williams Lane     path      1
                            Wimbo Park Shared Path cycleway  

In [18]:
import osmnx as ox, numpy as np, pandas as pd
import thesis_common as tc

SITES  = [65, 66, 74]
RADIUS = 120.0   # metres around the counter to list candidate edges

Gr   = tc.load_network()
snap = pd.read_csv(tc.DATA_PROCESSED / "count_link_snap.csv")
fit  = pd.read_csv(tc.DATA_PROCESSED / "stage3_link_fit.csv").set_index("SiteID")
X = {n: (Gr.nodes[n]["x"], Gr.nodes[n]["y"]) for n in Gr.nodes}

def _s(x):
    x = x[0] if isinstance(x, list) else x
    return "" if x is None else str(x)
def _lts(d):
    v = d.get("lts"); return int(v[0] if isinstance(v, list) else v)
def _brg(u, v):
    return round(np.degrees(np.arctan2(X[v][0]-X[u][0], X[v][1]-X[u][1])) % 360)

# calibrated edge flows (reuse if already built this session, else build from cached stores)
if "edge_flow" not in globals():
    cal = pd.read_csv(tc.DATA_PROCESSED / "od_calibrated_t.csv").sort_values("pair_idx").reset_index(drop=True)
    t_star = (cal["t_final"] if "t_final" in cal else cal["t"]).to_numpy(float)
    pi = float(tc.read_final()["pi_star"])
    exp_pairs, assembly, _ = tc.expand_pairs(cal)
    tc.set_costs(Gr, lam=tc.STRESS_PENALTY, beta=0.0, flat=False)
    sh = tc.route_store(tc.to_simple(Gr, "cost_high"), exp_pairs, tc.K_STORE, "cost_high", tc.store_tag("high"), n_jobs=1)
    sl = tc.route_store(tc.to_simple(Gr, "cost_low"),  exp_pairs, tc.K_STORE, "cost_low",  tc.store_tag("low"),  n_jobs=1)
    edge_flow = {}
    for w, st in [(pi * t_star, sl), ((1 - pi) * t_star, sh)]:
        for e, fv in tc.edge_flows_spread(st, tc.K_PATHS, w, assembly).items():
            edge_flow[e] = edge_flow.get(e, 0.0) + fv

for site in SITES:
    s = snap[snap.SiteID == site]
    if s.empty:
        print(f"Site {site}: not in count_link_snap.csv"); continue
    su, sv = int(s.iloc[0].u), int(s.iloc[0].v)
    sk = int(s.iloc[0].k) if "k" in s.columns else 0
    cmx, cmy = (X[su][0]+X[sv][0])/2, (X[su][1]+X[sv][1])/2
    obs = fit.loc[site, "v_obs"]   if site in fit.index else np.nan
    mod = fit.loc[site, "f_final"] if site in fit.index else np.nan
    print("=" * 84)
    print(f"Site {site}   obs {obs:.0f} / modelled {mod:.0f}   currently snapped to edge ({su}, {sv}, {sk})")
    rows = []
    for u, v, k, d in Gr.edges(keys=True, data=True):
        mx, my = (X[u][0]+X[v][0])/2, (X[u][1]+X[v][1])/2
        dist = np.hypot(mx - cmx, my - cmy)
        incident = (u in (su, sv)) or (v in (su, sv))
        if dist <= RADIUS or incident:
            rows.append(dict(u=u, v=v, k=k, name=_s(d.get("name")), hw=_s(d.get("highway")),
                             lts=_lts(d), length=round(d.get("len_m", 0)), brg=_brg(u, v),
                             dist=round(dist), flow=round(edge_flow.get((u, v), 0.0), 1),
                             matched=(u == su and v == sv), at_junction=incident))
    df = pd.DataFrame(rows).sort_values(["at_junction", "flow"], ascending=[False, False])
    print(df.to_string(index=False))

Site 65   obs 282 / modelled 26   currently snapped to edge (1838649886, 1838653623, 0)
          u           v  k                   name             hw  lts  length  brg  dist  flow  matched  at_junction
 1838649886    20958788  0        Moore Park Road      secondary    1      60  300    33 385.3    False         True
 1319751026  1838649886  0           Anzac Parade        primary    3      24  337    14 278.3    False         True
 1838649884  1838649886  0        Moore Park Road      secondary    3      15  287     6 188.1    False         True
 1838649886  1941652190  0        Flinders Street        primary    3     151  335    75  55.2    False         True
 1838653623  1838654608  0        Moore Park Road      secondary    3      68   91    39  26.1    False         True
 1838649886  1838653623  0        Moore Park Road      secondary    3      11   57     0  26.0     True         True
 2546390988  1838653623  0        Flinders Street        primary    3      48  156    25  10.

In [19]:
import numpy as np, pandas as pd
import thesis_common as tc

Gr   = tc.load_network()
snap = pd.read_csv(tc.DATA_PROCESSED / "count_link_snap.csv")
X = {n: (Gr.nodes[n]["x"], Gr.nodes[n]["y"]) for n in Gr.nodes}
sites = snap[snap.SiteID.isin([65, 66, 74])]
cx = np.mean([(X[int(r.u)][0]+X[int(r.v)][0])/2 for r in sites.itertuples()])
cy = np.mean([(X[int(r.u)][1]+X[int(r.v)][1])/2 for r in sites.itertuples()])

TAGS = ["highway","cycleway","cycleway:left","cycleway:right","cycleway:both",
        "segregated","foot","bicycle","width","surface","lanes","maxspeed","bridge"]
def _s(x):
    x = x[0] if isinstance(x, list) else x
    return "" if x is None else str(x)
def _lts(d):
    v = d.get("lts"); return int(v[0] if isinstance(v, list) else v)

rows = []
for u, v, k, d in Gr.edges(keys=True, data=True):
    mx, my = (X[u][0]+X[v][0])/2, (X[u][1]+X[v][1])/2
    if np.hypot(mx-cx, my-cy) > 700:
        continue
    if _s(d.get("highway")) in {"cycleway", "path", "footway"} and _lts(d) >= 2:   # separated but NOT low-stress
        rows.append(dict(name=_s(d.get("name")), lts=_lts(d), len=round(d.get("len_m", 0)),
                         **{t: _s(d.get(t)) for t in TAGS}))
df = pd.DataFrame(rows)
cols = ["name","lts","highway","cycleway","segregated","bicycle","width","surface","maxspeed","lanes","bridge"]
print(df[[c for c in cols if c in df]].sort_values(["name","lts"]).to_string(index=False))
print(f"\nseparated-ish edges rated LTS>=2 near Moore Park: {len(df)}  ({df.len.sum()/1000:.2f} km)")

                    name  lts  highway cycleway segregated    bicycle width  surface maxspeed lanes bridge
                            3 cycleway crossing         no designated        asphalt                      
                            3 cycleway crossing         no designated        asphalt                      
                            3 cycleway crossing         no designated        asphalt                      
                            3 cycleway crossing         no designated        asphalt                      
                            3 cycleway crossing            designated        asphalt                      
                            3 cycleway crossing            designated        asphalt                      
                            3 cycleway crossing            designated        asphalt                      
                            3 cycleway crossing            designated        asphalt                      
                            3 cyclewa

In [20]:
import thesis_common as tc
Gr = tc.load_network()

def dump(u, v):
    e = Gr.get_edge_data(u, v)
    if not e: print(f"  ({u},{v}): no edge"); return
    d = e[list(e)[0]]
    keys = ["name","highway","lts","cycleway","cycleway:left","cycleway:right","cycleway:both",
            "segregated","bicycle","foot","surface","width","maxspeed","lanes","oneway"]
    print("  " + "  ".join(f"{k}={d.get(k)}" for k in keys if d.get(k) is not None))

print("Site 65 — matched LTS3 edge vs adjacent LTS1 edge (both 'Moore Park Road'):")
dump(1838649886, 1838653623)   # matched, LTS 3, flow 26
dump(1838649886, 20958788)     # adjacent, LTS 1, flow 385
print("Site 74 — matched LTS3 edge vs adjacent LTS1 edge:")
dump(577292429, 577292433)     # matched, LTS 3, flow 4
dump(1845203361, 8113727666)   # adjacent, LTS 1, flow 272
print("Named pop-up cycleway / shared-path edges near Moore Park:")
for u, v, k, d in Gr.edges(keys=True, data=True):
    nm = d.get("name"); nm = nm[0] if isinstance(nm, list) else nm
    if nm and ("Pop-Up" in str(nm) or "Pop-up" in str(nm) or "Shared Path" in str(nm)):
        keys = ["name","highway","lts","cycleway","segregated","bicycle","surface","width","maxspeed"]
        print("  " + "  ".join(f"{kk}={d.get(kk)}" for kk in keys if d.get(kk) is not None))

Site 65 — matched LTS3 edge vs adjacent LTS1 edge (both 'Moore Park Road'):
  name=Moore Park Road  highway=secondary  lts=3  surface=asphalt  maxspeed=50  oneway=True
  name=Moore Park Road  highway=secondary  lts=1  surface=asphalt  maxspeed=50  lanes=2  oneway=True
Site 74 — matched LTS3 edge vs adjacent LTS1 edge:
  name=Moore Park Road  highway=secondary  lts=3  surface=asphalt  lanes=2  oneway=True
  name=Moore Park Road  highway=secondary  lts=1  surface=asphalt  maxspeed=40  lanes=3  oneway=True
Named pop-up cycleway / shared-path edges near Moore Park:
  name=Tempe Reserve Shared Path  highway=cycleway  lts=1  segregated=no  bicycle=designated  surface=concrete
  name=Alexandra Canal Shared Path  highway=cycleway  lts=3  cycleway=crossing  segregated=no  bicycle=designated  surface=concrete  width=3  maxspeed=50
  name=Alison Road Shared Path  highway=cycleway  lts=1  segregated=no  bicycle=designated  surface=asphalt  width=5
  name=Alison Road Shared Path  highway=cycleway  

In [21]:
import numpy as np, pandas as pd
import thesis_common as tc
Gr = tc.load_network()
snap = pd.read_csv(tc.DATA_PROCESSED / "count_link_snap.csv")
mem  = pd.read_csv(tc.DATA_PROCESSED / "count_link_members.csv")  # the corridor members per link
X = {n:(Gr.nodes[n]["x"],Gr.nodes[n]["y"]) for n in Gr.nodes}
def _name(u,v):
    d=Gr.get_edge_data(u,v); 
    return (tc._name(list(d.values())[0]) if d else None)

# for each station, how many distinct named parallel ways within 30 m share its road name but
# are NOT already corridor members?  (candidate missed carriageways)
for r in snap.itertuples():
    su,sv=int(r.u),int(r.v); nm=_name(su,sv)
    if not nm: continue
    cmx,cmy=(X[su][0]+X[sv][0])/2,(X[su][1]+X[sv][1])/2
    members=set(zip(mem.u,mem.v)) if {"u","v"}<=set(mem.columns) else set()
    missed=[]
    for u,v,k,d in Gr.edges(keys=True,data=True):
        if _name(u,v)==nm and (u,v) not in members:
            mx,my=(X[u][0]+X[v][0])/2,(X[u][1]+X[v][1])/2
            dd=np.hypot(mx-cmx,my-cmy)
            if 15 < dd <= 30: missed.append(round(dd))
    if missed:
        print(f"Site {int(r.SiteID)} '{nm}': {len(missed)} same-name edge(s) 15-30 m away not matched -> possible split carriageway")

Site 4 'Park Street': 2 same-name edge(s) 15-30 m away not matched -> possible split carriageway
Site 10 'Kent Street': 1 same-name edge(s) 15-30 m away not matched -> possible split carriageway
Site 16 'Macquarie Street': 2 same-name edge(s) 15-30 m away not matched -> possible split carriageway
Site 17 'St. James Road': 1 same-name edge(s) 15-30 m away not matched -> possible split carriageway
Site 19 'Bridge Street': 2 same-name edge(s) 15-30 m away not matched -> possible split carriageway
Site 21 'Harbour Bridge Cycleway': 2 same-name edge(s) 15-30 m away not matched -> possible split carriageway
Site 22 'York Street': 1 same-name edge(s) 15-30 m away not matched -> possible split carriageway
Site 26 'Bourke Road': 1 same-name edge(s) 15-30 m away not matched -> possible split carriageway
Site 27 'King Street': 3 same-name edge(s) 15-30 m away not matched -> possible split carriageway
Site 30 'Missenden Road': 4 same-name edge(s) 15-30 m away not matched -> possible split carriage

In [22]:
import numpy as np, pandas as pd
import thesis_common as tc
Gr = tc.load_network()
snap = pd.read_csv(tc.DATA_PROCESSED / "count_link_snap.csv")
mem  = pd.read_csv(tc.DATA_PROCESSED / "count_link_members.csv")
members = set(zip(mem.u.astype(int), mem.v.astype(int))) if {"u","v"} <= set(mem.columns) else set()
X = {n: (Gr.nodes[n]["x"], Gr.nodes[n]["y"]) for n in Gr.nodes}
def _name(u, v):
    d = Gr.get_edge_data(u, v); return tc._name(list(d.values())[0]) if d else None
def ang(u, v):
    return np.degrees(np.arctan2(X[v][0]-X[u][0], X[v][1]-X[u][1])) % 360
def angdiff(a, b):
    return abs((a - b + 180) % 360 - 180)

TOL_PAR, LAT_MIN, LAT_MAX, LON_MAX = 25, 8, 35, 20   # parallel bearing, lateral 8-35 m, near screenline
EL = []
for u, v, k, d in Gr.edges(keys=True, data=True):
    EL.append((u, v, _name(u, v), (X[u][0]+X[v][0])/2, (X[u][1]+X[v][1])/2, ang(u, v)))
EL = pd.DataFrame(EL, columns=["u","v","name","mx","my","brg"])

flags = []
for r in snap.itertuples():
    su, sv = int(r.u), int(r.v); nm = _name(su, sv)
    if not nm: continue
    b = ang(su, sv); M = np.array([(X[su][0]+X[sv][0])/2, (X[su][1]+X[sv][1])/2])
    br = np.radians(b); dvec = np.array([np.sin(br), np.cos(br)]); pvec = np.array([np.cos(br), -np.sin(br)])
    hits = []
    for e in EL[EL.name == nm].itertuples():
        if (int(e.u), int(e.v)) in members: continue
        w = np.array([e.mx - M[0], e.my - M[1]])
        lon, lat = abs(w @ dvec), abs(w @ pvec)
        par = angdiff(e.brg, b) <= TOL_PAR or angdiff(e.brg, (b+180) % 360) <= TOL_PAR
        if par and LAT_MIN <= lat <= LAT_MAX and lon <= LON_MAX:
            hits.append(round(lat))
    if hits:
        flags.append((int(r.SiteID), nm, len(hits), hits))

print(f"Stations with a genuine lateral (split) carriageway at the screenline: {len(flags)} / {len(snap)}")
for sid, nm, n, lats in flags:
    print(f"  Site {sid} '{nm}': {n} parallel edge(s), lateral offset {lats} m")

Stations with a genuine lateral (split) carriageway at the screenline: 7 / 77
  Site 26 'Bourke Road': 1 parallel edge(s), lateral offset [13] m
  Site 27 'King Street': 2 parallel edge(s), lateral offset [12, 13] m
  Site 35 'Regent Street': 1 parallel edge(s), lateral offset [8] m
  Site 66 'Cleveland Street': 1 parallel edge(s), lateral offset [13] m
  Site 67 'Alison Road': 1 parallel edge(s), lateral offset [12] m
  Site 74 'Moore Park Road': 1 parallel edge(s), lateral offset [13] m
  Site 100 'South Dowling Street': 3 parallel edge(s), lateral offset [20, 18, 21] m


### Node-spread artefact — Potts Point / Woolloomooloo (Discussion §5.3)

Diagnostic behind the Discussion's node-spreading limitation: the DZN straddles a ridge and a waterfront
the cyclable network only joins by a long detour, so some sampled sub-pairs are ~87 m apart straight-line
but ~1,137 m by road, producing the small circular flow in that corner of the Stage-3 flow map. The zone
carries ~2.4% of calibrated demand and sits away from the count stations.


In [23]:
# Node-spread circular-flow diagnostic for the Potts Point / Woolloomooloo zone
# (Discussion §5.3, Figure 4.8). Runs after the R2 cell, which builds store_high/low,
# assembly, rep_idx, exp_pairs, Gs_h, Gs_l; reuses Gr/cal/t_star/pi_star/edges (Gr is
# already projected, so node x/y are in metres). Produces the ~2.4%-of-demand figure and
# the 87 m vs 1,137 m straight-line-vs-road sub-pair cited in the thesis.
ZONE = "113331189"    # Potts Point / Woolloomooloo DZN: ridge + waterfront, split by the network
GAP_M, CIRC = 250.0, 1.8        # cluster gap (m); circuity flag (route/straight)

zone_reps = tc.load_zone_reps()
_reps = {str(k).strip(): [int(x) for x in v] for k, v in zone_reps.items()}
_repi = {}
for k, v in _reps.items():
    try: _repi[int(float(k))] = v
    except (ValueError, TypeError): pass
def reps_of(code, fb):
    s = str(code).strip()
    if s in _reps: return _reps[s]
    try:
        if int(float(s)) in _repi: return _repi[int(float(s))]
    except (ValueError, TypeError): pass
    return None if fb is None else [int(fb)]

exp_i = {(int(a), int(b)): i for i, (a, b) in enumerate(exp_pairs)}
def xy(n): return np.array([Gr.nodes[int(n)]["x"], Gr.nodes[int(n)]["y"]])   # metres
def rlen(Gs, p): return float(sum(Gs[int(a)][int(b)]["len_m"] for a, b in zip(p[:-1], p[1:])))

rep_nodes = reps_of(ZONE, None)
if rep_nodes is None:
    z7 = str(ZONE)[-7:]
    co, cd = cal["o"].astype(str).str.strip(), cal["d"].astype(str).str.strip()
    near = [k for k in zone_reps if str(ZONE) in str(k) or z7 in str(k)][:10]
    cn = sorted(set(co[co.str.contains(z7)]) | set(cd[cd.str.contains(z7)]))[:10]
    raise SystemExit(f"SA1 {ZONE} not in zone_rep_nodes.json ({len(zone_reps)} keys, "
                     f"first {list(zone_reps)[:3]}). keys~'{z7}': {near}; cal~'{z7}': {cn}")

# [A] geographic spread of the sampled rep nodes -> exclave / ridge straddle? ──
pts = np.array([xy(n) for n in rep_nodes])
D = np.hypot(*(pts[:, None, :] - pts[None, :, :]).transpose(2, 0, 1))
Gcl = nx.Graph(); Gcl.add_nodes_from(range(len(rep_nodes)))
for i in range(len(rep_nodes)):
    for j in range(i + 1, len(rep_nodes)):
        if D[i, j] <= GAP_M: Gcl.add_edge(i, j)
clusters = [sorted(c) for c in nx.connected_components(Gcl)]
scc = {n: k for k, c in enumerate(nx.strongly_connected_components(Gs_h)) for n in c}
def cl_of(node):
    for k, c in enumerate(clusters):
        if node in [rep_nodes[i] for i in c]: return k
    return -1
split = len(clusters) > 1
iu = np.triu_indices(len(D), 1)
print("=" * 74); print(f"SA1 {ZONE} — zone-spread circles diagnostic"); print("=" * 74)
print(f"\n[A] SAMPLE SPREAD: {len(rep_nodes)} rep nodes (N_REP={getattr(tc,'N_REP','?')}) {rep_nodes}")
print(f"    pairwise dist max {D[iu].max():,.0f} m | median {np.median(D[iu]):,.0f} m")
print(f"    clusters at {GAP_M:.0f} m gap: {len(clusters)}")
for k, c in enumerate(clusters):
    print(f"      cl{k}: {[rep_nodes[i] for i in c]} road-deg "
          f"{[Gs_h.degree(rep_nodes[i]) for i in c]} SCC {{{','.join(str(scc.get(rep_nodes[i])) for i in c)}}}")
if split:
    gaps = [D[np.ix_(a, b)].min() for i, a in enumerate(clusters) for b in clusters[i + 1:]]
    print(f"    >> SPLIT sample; nearest-cluster road gap {min(gaps):,.0f} m (exclave/ridge straddle)")
else:
    gaps = [0.0]; print(f"    single cluster (no straddle at {GAP_M:.0f} m)")

# [B] demand exposure ────────────────────────────────────────────────────────
co, cd = cal["o"].astype(str).str.strip(), cal["d"].astype(str).str.strip()
touch = (co == str(ZONE)) | (cd == str(ZONE))
pair_ids = np.where(touch.to_numpy())[0]
print(f"\n[B] DEMAND: {len(pair_ids)} O-D pairs touch this SA1 (of {len(cal):,}); "
      f"t*={t_star[touch.to_numpy()].sum():,.1f} "
      f"({100*t_star[touch.to_numpy()].sum()/t_star.sum():.2f}% of total)")

# [C] sub-pair route geometry: circuity, self-cross, gap-cross ────────────────
rows, seen = [], set()
for p in pair_ids:
    for a in reps_of(cal.loc[p, "o"], cal.loc[p, "o_node"]):
        for b in reps_of(cal.loc[p, "d"], cal.loc[p, "d_node"]):
            if a == b: continue
            si = exp_i.get((int(a), int(b)))
            if si is None or not store_high[si] or si in seen: continue
            seen.add(si)
            path = store_high[si][0]
            straight = np.hypot(*(xy(a) - xy(b)))
            ls = LineString([tuple(xy(n)) for n in path])
            rows.append(dict(sub=si, o=int(a), d=int(b), straight=straight,
                             route=rlen(Gs_h, path), circ=rlen(Gs_h, path) / max(straight, 1.0),
                             selfx=not ls.is_simple,
                             gapx=bool(split and cl_of(int(a)) != cl_of(int(b))
                                       and cl_of(int(a)) >= 0 and cl_of(int(b)) >= 0)))
sp = pd.DataFrame(rows)
print(f"\n[C] SUB-PAIR ROUTES: {len(sp)} unique routed for this SA1")
if len(sp):
    print(f"    circuity median {sp.circ.median():.2f} | p90 {sp.circ.quantile(.9):.2f} | max {sp.circ.max():.2f}")
    print(f"    circuity>{CIRC}: {int((sp.circ>CIRC).sum())} | self-intersecting {int(sp.selfx.sum())} "
          f"| gap-crossing {int(sp.gapx.sum())}")
    for r in sp.sort_values("circ", ascending=False).head(5).itertuples():
        print(f"      {r.o}->{r.d}: straight {r.straight:6,.0f} route {r.route:7,.0f} circ {r.circ:4.1f} "
              f"{'GAPX ' if r.gapx else ''}{'SELFX' if r.selfx else ''}")

# [D] aggregate loop signature: cyclomatic number of routed edge-union ────────
U = nx.Graph()
for r in sp.itertuples(): nx.add_path(U, [int(n) for n in store_high[r.sub][0]])
cyclo = U.number_of_edges() - U.number_of_nodes() + nx.number_connected_components(U)
print(f"\n[D] LOOP SIGNATURE: union nodes {U.number_of_nodes()} edges {U.number_of_edges()} "
      f"-> independent loops = {cyclo} {'(routes close into loops)' if cyclo else '(no loops)'}")

# [E] this SA1's exact contribution to the Stage-3 flow map ───────────────────
wz = np.where(touch.to_numpy(), pi_star * t_star, 0.0)
flz = tc.edge_flows_spread(store_low, tc.K_PATHS, wz, assembly)
for kk, x in tc.edge_flows_spread(store_high, tc.K_PATHS,
        np.where(touch.to_numpy(), (1 - pi_star) * t_star, 0.0), assembly).items():
    flz[kk] = flz.get(kk, 0.0) + x
print(f"\n[E] FLOW-MAP SHARE: {sum(flz.values()):,.0f} trip-edges on "
      f"{sum(1 for v in flz.values() if v > 0.5)} edges come from this one SA1")

# figure: the sub-pair routes + rep clusters ─────────────────────────────────
fig, ax = plt.subplots(figsize=(7.5, 7.5))
pad = 0.15 * max(np.ptp(pts[:, 0]), np.ptp(pts[:, 1])) + 400
allx = np.concatenate([pts[:, 0]] + [[xy(n)[0] for n in store_high[r.sub][0]] for r in sp.itertuples()]) if len(sp) else pts[:, 0]
ally = np.concatenate([pts[:, 1]] + [[xy(n)[1] for n in store_high[r.sub][0]] for r in sp.itertuples()]) if len(sp) else pts[:, 1]
ax.set_xlim(allx.min() - pad, allx.max() + pad); ax.set_ylim(ally.min() - pad, ally.max() + pad)
edges.plot(ax=ax, color="0.85", lw=0.3, zorder=0)
for r in sp.itertuples():
    xyp = np.array([xy(n) for n in store_high[r.sub][0]])
    ax.plot(xyp[:, 0], xyp[:, 1], lw=1.3, alpha=0.6, zorder=3,
            color="crimson" if (r.gapx or r.circ > CIRC) else "steelblue")
for k, c in enumerate(clusters):
    cp = np.array([xy(rep_nodes[i]) for i in c])
    ax.scatter(cp[:, 0], cp[:, 1], s=90, zorder=5, edgecolor="k",
               color=plt.cm.tab10(k), label=f"rep cluster {k} ({len(c)})")
ax.set_title(f"SA1 {ZONE}: {len(sp)} Stage-2 sub-pair routes\n"
             f"red = circuity>{CIRC} or gap-crossing; aggregate loops = {cyclo}", fontsize=10)
ax.set_aspect("equal"); ax.set_axis_off(); ax.legend(fontsize=8)
try: tc.save_report(fig, f"fig_diag_sa1_{ZONE}")
except Exception: fig.savefig(f"fig_diag_sa1_{ZONE}.png", dpi=150, bbox_inches="tight")
plt.show()

print("\n" + "=" * 74)
print(f"VERDICT: {len(rep_nodes)} rep nodes in {len(clusters)} cluster(s); "
      + (f"straddle ~{min(gaps):,.0f} m gap; " if split else "no straddle; ")
      + f"{int((sp.circ>CIRC).sum()) if len(sp) else 0}/{len(sp)} sub-pairs detoured, {cyclo} loop(s).")
print("=" * 74)

SA1 113331189 — zone-spread circles diagnostic

[A] SAMPLE SPREAD: 5 rep nodes (N_REP=5) [13887165, 1846062306, 4122585380, 9488372043, 9488372053]
    pairwise dist max 329 m | median 119 m
    clusters at 250 m gap: 1
      cl0: [13887165, 1846062306, 4122585380, 9488372043, 9488372053] road-deg [6, 6, 5, 2, 2] SCC {0,0,0,0,0}
    single cluster (no straddle at 250 m)

[B] DEMAND: 41 O-D pairs touch this SA1 (of 1,850); t*=91.1 (2.37% of total)

[C] SUB-PAIR ROUTES: 285 unique routed for this SA1
    circuity median 1.19 | p90 1.80 | max 13.01
    circuity>1.8: 28 | self-intersecting 5 | gap-crossing 0
      13711221288->1846062306: straight     87 route   1,137 circ 13.0 
      13711221286->1846062306: straight    160 route   1,055 circ  6.6 
      13711221289->1846062306: straight    177 route     993 circ  5.6 
      13711221288->9488372043: straight    263 route   1,054 circ  4.0 
      13711221288->9488372053: straight    333 route   1,111 circ  3.3 

[D] LOOP SIGNATURE: union n

In [24]:
# ── Appendix: full monitored-station list (Site ID, name, train/held-out) ─────
import pandas as pd, numpy as np, thesis_common as tc

links = pd.read_csv(tc.DATA_PROCESSED / "A_neutral_links.csv").sort_values("link_idx").reset_index(drop=True)
fit   = pd.read_csv(tc.DATA_PROCESSED / "stage3_link_fit.csv").sort_values("link_idx").reset_index(drop=True)
snap  = pd.read_csv(tc.DATA_PROCESSED / "count_link_snap.csv")
L = len(links)

assert {"SiteID", "split"} <= set(fit.columns), fit.columns.tolist()

names   = tc.link_names(links, snap)                                  # link_idx -> (street name, ...)
name_of = [names.get(i, (f"link {i}", None))[0] for i in range(L)]    # same labels as fig R6

stations = (pd.DataFrame({
        "SiteID":  fit["SiteID"].astype(int).to_numpy(),
        "Station": name_of,
        "Set":     np.where(fit["split"].eq("test"), "Held-out", "Training"),
    })
    .drop_duplicates("SiteID")
    .sort_values("SiteID")
    .reset_index(drop=True))

n_tr, n_te = (stations.Set == "Training").sum(), (stations.Set == "Held-out").sum()
print(f"{len(stations)} monitored stations ({n_tr} training / {n_te} held-out)\n")
print(stations.to_string(index=False))

csv_out = tc.FIG_REPORT / "appendix_station_list.csv"
stations.to_csv(csv_out, index=False)
print("\nsaved ->", csv_out)

# ── emit a ready-to-paste LaTeX longtable ─────────────────────────────────────
def _tex(s):
    for a, b in [("\\", r"\textbackslash "), ("&", r"\&"), ("%", r"\%"),
                 ("_", r"\_"), ("#", r"\#"), ("$", r"\$")]:
        s = str(s).replace(a, b)
    return s

body = "\n".join(f"{r.SiteID} & {_tex(r.Station)} & {r.Set} \\\\"
                 for r in stations.itertuples())

latex = rf"""\begin{{longtable}}{{r l l}}
\caption{{Complete list of the {len(stations)} monitored count stations with their
train/held-out assignment (70/30 stratified split, frozen before the single
held-out evaluation).}}\label{{tab:station_list}}\\
\toprule
\textbf{{Site ID}} & \textbf{{Station}} & \textbf{{Set}} \\
\midrule
\endfirsthead
\multicolumn{{3}}{{@{{}}l}}{{\textit{{Table~\ref{{tab:station_list}} continued}}}} \\
\toprule
\textbf{{Site ID}} & \textbf{{Station}} & \textbf{{Set}} \\
\midrule
\endhead
\midrule
\multicolumn{{3}}{{r@{{}}}}{{\textit{{continued on next page}}}} \\
\endfoot
\bottomrule
\endlastfoot
{body}
\end{{longtable}}"""

tex_out = tc.FIG_REPORT / "appendix_station_list.tex"
tex_out.write_text(latex)
print("\nsaved ->", tex_out)
print("\n% ---- paste into app:stations (or \\input the .tex) ----\n")
print(latex)

77 monitored stations (53 training / 24 held-out)

 SiteID                        Station      Set
      2   Goulburn St / Castlereagh St Training
      4         Park St / Elizabeth St Training
      5     Elizabeth St / Campbell St Held-out
      8       Wynyard La / Margaret St Held-out
     10            Kent St / Druitt St Training
     11              Kent St / King St Training
     12   Liverpool St / Wentworth Ave Training
     13       Sussex St / Liverpool St Held-out
     14                   Liverpool St Held-out
     16         Macquarie St / Bent St Held-out
     17    St. James Rd / Macquarie St Training
     19            Bridge St / Pitt St Held-out
     21            Harbour Br Cycleway Training
     22 York St / Bradfield Hwy Onramp Training
     23          Maddox St / Bourke Rd Training
     25                        Site 25 Training
     26          Bourke St / Bourke Rd Training
     27       King St / Sydney Park Rd Training
     28        Bowden St / Mandible S